# DR-VERGE Enhanced — Supplementary Mechanism Analysis (inference-only)

**Companion to** `full_pipeline_notebook_enhanced.ipynb` (run `enhanced_v1_20260811`, 11 Aug 2026).
This notebook deepens the frozen Enhanced run **without training, tuning, re-selecting or overwriting
anything**. It re-infers the three CORAL heads (`p_dual`, `p_macula`, `p_disc`) of the frozen teacher
and the 20 frozen core students (4 conditions × 5 seeds), reproduces the published numbers, and only
then adds new, explicitly **exploratory** inferential evidence.

| Phase | What | Blocking? |
|---|---|---|
| **P0** | Drive discovery, inventory, SHA-256 manifest, locked-config check, teacher + CSD-s42 smoke reproduction | yes — stops the notebook |
| **P1** | 20 students on DRTiD test (550 eyes): reproduce QWK + ShiftL1/CosAgree/BenefitCorr, then 9 paired CSD-vs-baseline contrasts (hierarchical paired bootstrap B=10,000) | yes |
| **P2** | Same frozen models on DeepDRiD Set-C (100 patients / 200 eyes / 400 images), patient-clustered paired bootstrap — **post-hoc exploratory** | runs only if P1 passed; a failure is reported, not raised |
| **P3** | Algebraic decomposition, per-grade errors, teacher-correct/incorrect subgroups — from the cache, no new inference | yes (identity check) |
| **P4** | Paper-ready tables/figures/paragraphs (Indonesian), claim ledger, audit, source-unchanged check | yes |

**Safety rules enforced in code**
* No training function exists in this notebook; gradients are globally disabled; a gate checks that no optimizer object was ever created.
* Every write goes through a guard that refuses any path inside the original artifact tree.
* Original checkpoints and reference files are SHA-256 hashed at the start and re-hashed at the end.
* Decision thresholds are read from the original `threshold_calibration.json` and must equal the locked values — never re-tuned.
* Mechanism definitions are copied from the source notebook (`shift_fidelity`, `_nll`, `delta`) and cross-checked by an independent implementation.

**How to run:** `Runtime → Run all` (GPU recommended, CPU works). Re-running is cheap: per-model inference is cached
with schema + checksum metadata and reused only when the checkpoint hash and data fingerprint match.

In [ ]:
# ---- 0. Runtime: Drive + the two extra packages the source notebook also pinned ----
import os, sys, subprocess
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")      # same as the source notebook
os.environ.setdefault("NO_ALBUMENTATIONS_UPDATE", "1")
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Same albumentations pin as the source run, so Resize/Normalize are bit-compatible.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "albumentations==1.4.21",
                    "openpyxl>=3.1"], check=False)
print("Colab:", IN_COLAB)

## 1 — Configuration
Only the first block is meant to be edited. Everything below it is **locked** from the source run and is
verified against the original artifact tree at runtime (a mismatch stops the notebook).

In [ ]:
import json, glob, time, math, hashlib, gc, copy, warnings, platform
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
torch.set_grad_enabled(False)          # inference-only notebook: autograd is OFF globally

# ---------------- EDIT ONLY THESE ----------------
DRIVE_BASE   = os.environ.get("DRVERGE_DRIVE_BASE", "/content/drive/MyDrive/DR-VERGE")
ORIG_RUN_TAG = "enhanced_v1_20260811"                  # the frozen Enhanced run (read-only)
SUPP_TAG     = "enhanced_supplementary_v1_20261029"    # new, separate output tree
INFER_DEVICE = "auto"        # "auto" | "cuda" | "cpu"
RUN_EXTERNAL = True          # P2 (DeepDRiD Set-C) -- runs only after P1 passes
FORCE_REINFER = False        # True ignores the inference cache (never touches original files)
B_PILOT, B_FINAL = 1000, 10000   # pilot run checks the pipeline/speed; FINAL is what gets reported
P_PERM = 10000               # conditional permutation draws (secondary evidence)
RNG_SEED = 20261029          # every bootstrap/permutation draw derives from this
# -------------------------------------------------

# ================= LOCKED (verified against the original artifacts below) =================
CORE_CONDITIONS = ["dual_no_distill", "dual_logitkd", "dual_featkd", "dual_csd"]
BASELINES = ["dual_no_distill", "dual_logitkd", "dual_featkd"]
CSD = "dual_csd"
LABEL = {"teacher": "Teacher", "dual_no_distill": "NoDistill", "dual_logitkd": "Logit-KD",
         "dual_featkd": "Feature-KD", "dual_csd": "CSD"}
SEEDS_CORE = [42, 123, 2026, 3407, 8888]
IMG_SIZE, SAMPLING = 384, "standard"
NUM_CLASSES = 5
NUM_THRESHOLDS = NUM_CLASSES - 1
STUDENT_CHANNELS = (32, 64, 96, 128, 160, 192, 224)
DRTID_MEAN, DRTID_STD = [0.372487, 0.217266, 0.119367], [0.281526, 0.179457, 0.109162]
LOCKED_THRESHOLDS = {"dual_no_distill": 0.40, "dual_logitkd": 0.50, "dual_featkd": 0.50, "dual_csd": 0.50}
TEACHER_THRESHOLD = 0.5      # source thr_for("teacher") falls through to 0.5 (teacher is never calibrated)
LOCKED_HPARAMS = {"logitkd": {"alpha": 0.5, "tau_kd": 2.0}, "featkd": {"gamma_feat": 2.0},
                  "csd_variant": "smoothl1_norm", "csd_beta": 0.1}
SELECTED_METHOD = "dual_csd"           # M*, validation-only; NOT re-selected here
INTERNAL_BATCH, EXTERNAL_BATCH = 8, 16  # same loaders as the source (eval-mode BN: batch-invariant)
N_INTERNAL, N_EXT_EYES, N_EXT_PATIENTS = 550, 200, 100
DEEPDRID_PRIMARY_ORDER = "_1=macula"   # pre-registered field order of the source run
METRICS = ["ShiftL1", "CosAgree", "BenefitCorr"]
BETTER = {"ShiftL1": "lower", "CosAgree": "higher", "BenefitCorr": "higher",
          "BenefitCorrSpearman": "higher"}
# SHA-256 of the split CSVs exactly as recorded by the source run in configs/split_manifest.json
EXPECTED_SPLIT_SHA256 = {
    "train": "4384272e5455d75ef1e5e49dc6ba8a700de4ce97efd0c8809622ed649e87c5fa",
    "val":   "66609c6786621311a56e1207b11176c59dc7108e0f52b9a6c2361fb35bca3529",
    "test":  "6191ea9808fac2133e0efc9052dfa62309ceabf5293cd079a3a20e2cbcd12124"}
REPO_SOURCE_NOTEBOOK_SHA256 = "72398b3377ae195796b9a359aeee97b4884c51c6bd19246cc0a1277a2b95f1cb"
# Published 4-decimal values (paper Tables I/III/V). Used ONLY as a second reproduction reference;
# the primary reference is the full-precision per-seed file of the source run.
PAPER_REFERENCE = {
    "teacher_QWK": 0.7364,
    "QWK":         {"dual_no_distill": 0.6042, "dual_logitkd": 0.5942, "dual_featkd": 0.6161, "dual_csd": 0.6018},
    "ShiftL1":     {"dual_no_distill": 0.3759, "dual_logitkd": 0.3840, "dual_featkd": 0.3718, "dual_csd": 0.3509},
    "CosAgree":    {"dual_no_distill": 0.3509, "dual_logitkd": 0.2858, "dual_featkd": 0.3815, "dual_csd": 0.4361},
    "BenefitCorr": {"dual_no_distill": 0.2193, "dual_logitkd": 0.1795, "dual_featkd": 0.1943, "dual_csd": 0.3075},
    "ext_teacher_QWK": 0.7923, "ext_csd_QWK": 0.6688,
    "dQWK":        {"dual_no_distill": -0.0024, "dual_logitkd": 0.0077, "dual_featkd": -0.0143}}
# Reproduction tolerances (absolute). A different GPU (TF32 on A100 vs plain FP32 elsewhere) moves
# probabilities by ~1e-4 at most; one near-threshold grade flip among 550 eyes moves a seed QWK by ~2e-3.
# A real preprocessing/model error moves probabilities by >=1e-2 on average and is always caught. A QWK gap is
# accepted ONLY when every changed grade sits on an eye whose saved probability is within `flip_margin` of the frozen
# threshold (pure floating-point tie-breaking) and such eyes are at most 2% of the records.
TOL = {"prob_max": 1e-2, "prob_mean": 5e-4, "grade_agree_min": 0.995, "flip_margin": 5e-3, "flip_max_frac": 0.02,
       "mech_seed": 2e-3, "mech_mean": 5e-4, "qwk_seed": 5e-3, "qwk_mean": 1e-3,
       "paper_round": 5e-4, "impl_crosscheck": 5e-5,
       "ext_grade_agree_min": 0.99, "ext_qwk_seed": 1e-2, "ext_qwk_mean": 2e-3}
SCHEMA_VERSION = "drverge-supp-cache-v1"
# ==========================================================================================

def _find_drtid(root):
    for c in (f"{root}/DRTiD/DRTiD", f"{root}/DRTiD"):
        if os.path.exists(f"{c}/Ground Truths/DR_grade/a. DR_grade_Training.csv"):
            return c
    return None

def _find_deepdrid(root):
    for c in (f"{root}/DeepDRiD-master/regular_fundus_images", f"{root}/DeepDRiD/regular_fundus_images",
              f"{root}/DeepDRiD-master", f"{root}/DeepDRiD"):
        if os.path.exists(f"{c}/regular-fundus-validation/regular-fundus-validation.csv"):
            return c
    return None

def discover_original_artifacts():
    """The source tree is DRIVE_BASE/artifacts_<RUN_TAG>. Verified on disk, never assumed."""
    want = f"{DRIVE_BASE}/artifacts_{ORIG_RUN_TAG}"
    if os.path.isdir(want):
        return want, []
    # Look one and two levels around MyDrive before giving up -- but only for the SAME run tag.
    roots = {os.path.dirname(DRIVE_BASE.rstrip("/")), DRIVE_BASE}
    cands = sorted({p for r in roots for p in
                    glob.glob(f"{r}/artifacts_{ORIG_RUN_TAG}") + glob.glob(f"{r}/*/artifacts_{ORIG_RUN_TAG}")
                    if os.path.isdir(p)})
    others = sorted(glob.glob(f"{DRIVE_BASE}/artifacts_*"))
    return (cands[0] if len(cands) == 1 else None), (cands if len(cands) != 1 else []) + others

ORIG_ART, _alts = discover_original_artifacts()
if ORIG_ART is None:
    raise RuntimeError(f"Original artifact tree artifacts_{ORIG_RUN_TAG} NOT found under {DRIVE_BASE}. "
                       f"Candidates seen: {_alts}. Set DRIVE_BASE correctly -- nothing will be retrained.")
if os.path.abspath(os.path.dirname(ORIG_ART)) != os.path.abspath(DRIVE_BASE):
    print(f"NOTE: original run found at {ORIG_ART} (not directly under DRIVE_BASE); using its parent.")
    DRIVE_BASE = os.path.dirname(ORIG_ART)
SUPP_ART = f"{DRIVE_BASE}/artifacts_{SUPP_TAG}"
if os.path.abspath(SUPP_ART).startswith(os.path.abspath(ORIG_ART)):
    raise RuntimeError("SUPP_ART must not live inside the original artifact tree")
DATASET_ROOT = f"{DRIVE_BASE}/dataset"
DRTID_ROOT = _find_drtid(DATASET_ROOT)
DRTID_IMAGE_ROOT = f"{DRTID_ROOT}/Original Images" if DRTID_ROOT else None
DEEPDRID_ROOT = _find_deepdrid(DATASET_ROOT)

O_CKPT, O_CFG, O_SPLIT = f"{ORIG_ART}/checkpoints", f"{ORIG_ART}/configs", f"{ORIG_ART}/splits"
O_RES = f"{ORIG_ART}/results"
O_MET, O_TAB, O_PRED = f"{O_RES}/metrics", f"{O_RES}/tables", f"{O_RES}/predictions"
TEACHER_CKPT = f"{O_CKPT}/teacher_{IMG_SIZE}_{SAMPLING}.pt"
def student_ckpt(cond, seed): return f"{O_CKPT}/student/{cond}/seed{seed}.pt"

LOGS, CACHE, TABLES = f"{SUPP_ART}/logs", f"{SUPP_ART}/cache", f"{SUPP_ART}/tables"
FIGS, REPORTS = f"{SUPP_ART}/figures", f"{SUPP_ART}/reports"
for _d in (SUPP_ART, LOGS, CACHE, TABLES, FIGS, REPORTS, f"{CACHE}/internal", f"{CACHE}/external",
           f"{LOGS}/pilot"):
    os.makedirs(_d, exist_ok=True)

print("original (read-only):", ORIG_ART)
print("supplementary out   :", SUPP_ART)
print("DRTiD               :", DRTID_ROOT)
print("DeepDRiD            :", DEEPDRID_ROOT)

## 2 — Guarded I/O, gates and stage reports
Every writer refuses paths inside the original artifact tree and writes atomically (temp file → rename),
so an interrupted session can never leave a half-written table that a later run silently reuses.

In [ ]:
class SupplementaryBlocked(RuntimeError):
    pass

_ORIG_ABS, _SUPP_ABS = os.path.abspath(ORIG_ART), os.path.abspath(SUPP_ART)

def _guard(path):
    ap = os.path.abspath(path)
    if ap == _ORIG_ABS or ap.startswith(_ORIG_ABS + os.sep):
        raise SupplementaryBlocked(f"REFUSED: write inside the original artifact tree: {path}")
    if not ap.startswith(_SUPP_ABS + os.sep):
        raise SupplementaryBlocked(f"REFUSED: write outside the supplementary tree: {path}")
    os.makedirs(os.path.dirname(ap), exist_ok=True)
    return ap

def _atomic(path, writer):
    ap = _guard(path)
    tmp = ap + ".tmp"
    for i in range(4):                    # Drive's FUSE mount occasionally rejects a write
        try:
            writer(tmp); os.replace(tmp, ap); return ap
        except OSError as e:
            if i == 3: raise
            print(f"  write retry {i+1}: {e}"); time.sleep(1.5 * (i + 1))

def _jsonable(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)

def write_text(path, text):
    def w(p):
        with open(p, "w", encoding="utf-8") as f: f.write(text)
    return _atomic(path, w)

def write_json(obj, path):
    return write_text(path, json.dumps(obj, indent=2, default=_jsonable))

def write_csv(df, path, **kw):
    return _atomic(path, lambda p: df.to_csv(p, index=False, **kw))

try:
    import pyarrow  # noqa: F401
    HAVE_PARQUET = True
except ImportError:
    HAVE_PARQUET = False

def write_table(df, base):
    """Parquet when available (exact float32 round-trip), CSV fallback with full float repr."""
    if HAVE_PARQUET:
        return _atomic(base + ".parquet", lambda p: df.to_parquet(p, index=False))
    return _atomic(base + ".csv", lambda p: df.to_csv(p, index=False, float_format="%.9g"))

def read_table(base):
    if os.path.exists(base + ".parquet") and HAVE_PARQUET: return pd.read_parquet(base + ".parquet")
    if os.path.exists(base + ".csv"): return pd.read_csv(base + ".csv", dtype={"sample_id": str})
    raise FileNotFoundError(base)

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()

def sha256_lf(path):
    """Hash with CRLF normalised to LF -- tolerates a CSV that passed through a Windows checkout."""
    with open(path, "rb") as f: data = f.read()
    return hashlib.sha256(data.replace(b"\r\n", b"\n")).hexdigest()

def sha256_bytes(b): return hashlib.sha256(b).hexdigest()

GATES = []
def gate(name, passed, detail="", blocking=True, phase="P0"):
    GATES.append({"phase": phase, "gate": name, "passed": bool(passed), "blocking": bool(blocking),
                  "detail": str(detail), "time": time.strftime("%Y-%m-%dT%H:%M:%S")})
    print(f"{'PASS' if passed else ('FAIL' if blocking else 'WARN')} | {phase} | {name}"
          + (f" | {detail}" if detail else ""))
    write_csv(pd.DataFrame(GATES), f"{LOGS}/gate_report.csv")
    if blocking and not passed:
        write_text(f"{REPORTS}/BLOCKED_{phase}_{name}.md",
                   f"# BLOCKED at {phase} / {name}\n\n{detail}\n\nNothing was retrained or tuned. "
                   "Trace the mismatch in logs/reproduction_checks.csv before changing anything.\n")
        raise SupplementaryBlocked(f"{phase}/{name}: {detail}")
    return bool(passed)

PHASE_STATUS = {"P0": "NOT_RUN", "P1": "NOT_RUN", "P2": "NOT_RUN", "P3": "NOT_RUN", "P4": "NOT_RUN"}
STAGE_LOG = []
def stage_report(phase, title, lines):
    STAGE_LOG.append({"phase": phase, "title": title, "lines": list(lines)})
    md = "\n\n".join(f"## {s['phase']} — {s['title']}\n\n" + "\n".join(f"- {l}" for l in s["lines"])
                     for s in STAGE_LOG)
    write_text(f"{REPORTS}/STAGE_LOG.md", "# Stage log\n\n" + md + "\n")

def remove_stale(paths):
    """Delete earlier outputs of THIS supplementary tree only (guarded), so a re-run never leaves
    a stale completion marker or a stale external table next to fresh results."""
    for p in paths:
        if os.path.exists(p):
            os.remove(_guard(p))

# completion markers / blocker notes are per run
remove_stale(glob.glob(f"{LOGS}/SUPPLEMENTARY_RUN_*.txt") + glob.glob(f"{REPORTS}/BLOCKED_*.md"))
print("guarded I/O ready | parquet:", HAVE_PARQUET)

## 3 — Frozen model, data and metric definitions (copied from the source notebook)
Copied **verbatim** from `full_pipeline_notebook_enhanced.ipynb` §6, §7, §9 and the evaluation helpers,
minus everything that trains. One deliberate difference: `tv.resnet50(weights=None)` instead of downloading
ImageNet weights — every teacher parameter and buffer is overwritten by `load_state_dict(strict=True)`,
so the result is identical and no network access is needed.

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import torchvision, torchvision.models as tv
from sklearn.metrics import (confusion_matrix, f1_score, precision_recall_fscore_support,
                             balanced_accuracy_score, cohen_kappa_score)
from scipy import stats as sps

# ---- models (source §7) ----
class CORALHead(nn.Module):
    def __init__(self, in_dim, num_classes=NUM_CLASSES, init=None):
        super().__init__()
        self.k = num_classes - 1
        self.fc = nn.Linear(in_dim, 1, bias=False)
        t = torch.tensor(list(init if init is not None else [-0.7 * i for i in range(self.k)]),
                         dtype=torch.float32)
        gaps = (t[:-1] - t[1:]).clamp_min(1e-4)
        self.base_bias = nn.Parameter(t[0].clone())
        self.bias_steps = nn.Parameter(torch.log(torch.expm1(gaps)).clone())
    def biases(self):
        steps = F.softplus(self.bias_steps)
        return self.base_bias - torch.cat([torch.zeros(1, device=steps.device), torch.cumsum(steps, 0)])
    def forward(self, z):
        logits = self.fc(z) + self.biases().unsqueeze(0)
        return logits, torch.sigmoid(logits)

class InteractionFusion(nn.Module):
    def __init__(self, feat_dim, hidden=None):
        super().__init__()
        hidden = hidden or feat_dim
        self.norm_in  = nn.LayerNorm(feat_dim * 4)
        self.proj     = nn.Linear(feat_dim * 4, hidden)
        self.act      = nn.ReLU(inplace=True)
        self.norm_out = nn.LayerNorm(hidden)
        self.out_dim  = hidden
    def forward(self, zm, zd):
        return self.norm_out(self.act(self.proj(
            self.norm_in(torch.cat([zm, zd, torch.abs(zm - zd), zm * zd], dim=1)))))

class DWSepBlock(nn.Module):
    def __init__(self, i, o, stride=1):
        super().__init__()
        self.dw, self.bn1, self.act1 = nn.Conv2d(i, i, 3, stride, 1, groups=i, bias=False), nn.BatchNorm2d(i), nn.ReLU(inplace=True)
        self.pw, self.bn2, self.act2 = nn.Conv2d(i, o, 1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True)
    def forward(self, x): return self.act2(self.bn2(self.pw(self.act1(self.bn1(self.dw(x))))))

class LightBackbone(nn.Module):
    def __init__(self, channels=STUDENT_CHANNELS):
        super().__init__()
        ch = tuple(channels)
        self.stem_conv = nn.Conv2d(3, ch[0], 3, 2, 1, bias=False)
        self.stem_bn, self.stem_act = nn.BatchNorm2d(ch[0]), nn.ReLU(inplace=True)
        strides = [2 if i % 2 == 0 else 1 for i in range(len(ch) - 1)]
        self.blocks = nn.ModuleList([DWSepBlock(ch[i], ch[i + 1], strides[i]) for i in range(len(ch) - 1)])
        self.gap, self.out_dim = nn.AdaptiveAvgPool2d(1), ch[-1]
    def forward(self, x):
        x = self.stem_act(self.stem_bn(self.stem_conv(x)))
        for b in self.blocks: x = b(x)
        return self.gap(x).flatten(1)

class DualViewBase(nn.Module):
    def forward(self, macula, disc):
        zm, zd = self.backbone(macula), self.backbone(disc)
        zf = self.fusion(zm, zd)
        ld, pd_ = self.main_head(zf)
        lm, pm = self.macula_head(zm)
        ldd, pdd = self.disc_head(zd)
        return {"p_dual": pd_, "logit_dual": ld, "p_macula": pm, "logit_macula": lm,
                "p_disc": pdd, "logit_disc": ldd, "z_fused": zf}

class Teacher(DualViewBase):
    def __init__(self, feat_dim=2048, init=None):
        super().__init__()
        bb = tv.resnet50(weights=None); bb.fc = nn.Identity()   # weights come from the checkpoint
        self.backbone = bb
        self.fusion = InteractionFusion(feat_dim)
        self.main_head = CORALHead(self.fusion.out_dim, NUM_CLASSES, init)
        self.macula_head = CORALHead(feat_dim, NUM_CLASSES, init)
        self.disc_head = CORALHead(feat_dim, NUM_CLASSES, init)

class Student(DualViewBase):
    def __init__(self, backbone=None, init=None):
        super().__init__()
        self.backbone = backbone or LightBackbone()
        fd = self.backbone.out_dim
        self.fusion = InteractionFusion(fd)
        self.main_head = CORALHead(self.fusion.out_dim, NUM_CLASSES, init)
        self.macula_head = CORALHead(fd, NUM_CLASSES, init)
        self.disc_head = CORALHead(fd, NUM_CLASSES, init)

def param_count(m): return int(sum(p.numel() for p in m.parameters()))

# ---- data (source §6, evaluation path only: no augmentation exists here) ----
IMAGE_CACHE, _RESIZERS = {}, {}
def load_img(path, size=None):
    size = size or IMG_SIZE
    key = (path, size)
    a = IMAGE_CACHE.get(key)
    if a is None:
        r = _RESIZERS.get(size)
        if r is None: r = _RESIZERS[size] = A.Resize(size, size)
        a = r(image=np.array(Image.open(path).convert("RGB")))["image"]
        IMAGE_CACHE[key] = a
    return a

def prefill_cache(paths, label, workers=8, chunk=32):
    """Parallel JPEG decode (I/O bound on Drive); resize stays in the main thread exactly as load_img
    does it, so the cached arrays are identical to serial decoding. Chunked to bound memory."""
    size = IMG_SIZE
    r = _RESIZERS.get(size)
    if r is None: r = _RESIZERS[size] = A.Resize(size, size)
    todo = [p for p in dict.fromkeys(paths) if (p, size) not in IMAGE_CACHE]
    dec = lambda p: np.array(Image.open(p).convert("RGB"))
    with ThreadPoolExecutor(workers) as ex:
        for i in tqdm(range(0, len(todo), chunk), desc=f"decoding {label}", leave=False):
            part = todo[i:i + chunk]
            for p, arr in zip(part, ex.map(dec, part)):
                IMAGE_CACHE[(p, size)] = r(image=arr)["image"]
    print(f"  {label}: {len(todo)} decoded, cache {len(IMAGE_CACHE)} arrays")

def build_tf(train, mean, std, size=None):
    assert not train, "this notebook never builds a training transform"
    return A.Compose([A.Resize(size or IMG_SIZE, size or IMG_SIZE), A.Normalize(mean=mean, std=std), ToTensorV2()])

eval_tf = build_tf(False, DRTID_MEAN, DRTID_STD)

class DualViewDataset(Dataset):
    def __init__(self, csv, transform=None, image_root=None, size=None):
        self.size = size or IMG_SIZE
        self.df = pd.read_csv(csv)
        root = image_root or DRTID_IMAGE_ROOT
        self.df["macula_path"] = self.df["macula_filename"].apply(lambda f: f"{root}/{f}")
        self.df["disc_path"]   = self.df["disc_filename"].apply(lambda f: f"{root}/{f}")
        self.transform = transform or eval_tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        m, d = load_img(r["macula_path"], self.size), load_img(r["disc_path"], self.size)
        a, b = self.transform(image=m)["image"], self.transform(image=d)["image"]
        return {"macula": a, "disc": b, "label": torch.tensor(int(r["grade"]), dtype=torch.long),
                "cluster_id": int(r["record_id"]), "sample_id": str(r["record_id"])}

class SetCDataset(Dataset):
    """Verbatim from source §19: one label row per IMAGE; patient/eye/field parsed from image_id."""
    SUB = "Online-Challenge1&2-Evaluation"
    def __init__(self, root, transform=None, order=DEEPDRID_PRIMARY_ORDER):
        self.tf, self.order = transform or eval_tf, order
        base = f"{root}/{self.SUB}"; lab = f"{base}/Challenge1_labels.xlsx"
        self.available = os.path.exists(lab)
        recs, excl = [], []
        if not self.available:
            self.df, self.excl = pd.DataFrame(), pd.DataFrame(); return
        df = pd.read_excel(lab)
        gcol = "DR_Levels" if "DR_Levels" in df.columns else df.columns[-1]
        df["image_id"] = df["image_id"].astype(str)
        df["pid"]   = df["image_id"].str.split("_").str[0]
        df["eye"]   = df["image_id"].str.extract(r"_([lr])")[0]
        df["field"] = df["image_id"].str.extract(r"_[lr](\d+)$")[0]
        disk = {}
        for dp, _, fns in os.walk(f"{base}/Images"):
            for fn in fns:
                if fn.lower().endswith((".jpg", ".jpeg", ".png")):
                    disk.setdefault(os.path.splitext(fn)[0], os.path.join(dp, fn))
        for (pid, eye), g in df.groupby(["pid", "eye"]):
            def drop(reason, grade=None):
                excl.append({"patient_id": pid, "eye": eye, "reason": reason,
                             "n_views": len(g), "image_ids": ",".join(g["image_id"]), "grade": grade})
            fields = sorted(g["field"].dropna().astype(str))
            if set(fields) != {"1", "2"}:
                drop("MISSING_FIELD" if len(fields) < 2 else "UNEXPECTED_FIELD_COUNT"); continue
            lv = g[gcol].dropna()
            if lv.empty: drop("MISSING_GRADE"); continue
            if lv.nunique() > 1: drop("INCONSISTENT_GRADE", ";".join(map(str, lv.unique()))); continue
            grade = int(lv.iloc[0])
            if not 0 <= grade < NUM_CLASSES: drop("INVALID_GRADE", grade); continue
            ids = g.sort_values("field")["image_id"].tolist()
            paths = [disk.get(i) for i in ids]
            if any(p is None for p in paths): drop("PATH_NOT_FOUND", grade); continue
            recs.append({"patient_id": int(pid), "eye": eye, "f1": paths[0], "f2": paths[1],
                         "grade": grade})
        self.df, self.excl = pd.DataFrame(recs), pd.DataFrame(excl)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        a, b = (r["f1"], r["f2"]) if self.order == "_1=macula" else (r["f2"], r["f1"])
        return {"macula": self.tf(image=load_img(a))["image"],
                "disc": self.tf(image=load_img(b))["image"],
                "label": torch.tensor(int(r["grade"]), dtype=torch.long),
                "cluster_id": int(r["patient_id"]), "sample_id": f"{int(r['patient_id'])}_{r['eye']}"}

def _ids(batch):
    v = batch.get("sample_id")
    if v is None: return [str(c) for c in batch["cluster_id"].tolist()]
    return [str(x) for x in (v.tolist() if torch.is_tensor(v) else v)]

# ---- metrics (source §9) ----
def grade_from(p_cum, t):
    """CORAL rule: grade = COUNT of cumulative outputs strictly above the frozen threshold t."""
    p = p_cum if isinstance(p_cum, np.ndarray) else np.asarray(p_cum)
    return (p > t).sum(1)

def fast_qwk(y_true, y_pred, K=NUM_CLASSES):
    y_true, y_pred = np.asarray(y_true, int), np.asarray(y_pred, int)
    O = np.zeros((K, K)); np.add.at(O, (y_true, y_pred), 1)
    w = (np.arange(K)[:, None] - np.arange(K)[None, :]) ** 2 / (K - 1) ** 2
    E = np.outer(np.bincount(y_true, minlength=K), np.bincount(y_pred, minlength=K)) / max(len(y_true), 1)
    den = (w * E).sum()
    return 1.0 - (w * O).sum() / den if den > 0 else 0.0

def all_metrics(y_true, y_pred, K=NUM_CLASSES):
    y_true, y_pred = np.asarray(y_true, int), np.asarray(y_pred, int)
    labels = list(range(K))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    pr, rc, f1, sup = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    m = {"QWK": fast_qwk(y_true, y_pred), "Accuracy": float((y_true == y_pred).mean()),
         "BalancedAccuracy": float(balanced_accuracy_score(y_true, y_pred)),
         "MacroF1": float(f1_score(y_true, y_pred, average="macro", labels=labels, zero_division=0)),
         "MAE": float(np.mean(np.abs(y_true - y_pred))),
         "SevereErrorRate": float(np.mean(np.abs(y_true - y_pred) >= 2))}
    for g in labels:
        m[f"Recall_Grade{g}"] = float(rc[g]); m[f"Support_Grade{g}"] = int(sup[g])
    return m, cm

# ---- mechanism definitions (source _nll / delta / shift_fidelity) ----
def delta(p_dual, p_m, p_d): return p_dual - (p_m + p_d) / 2

def _nll(p_cum, y):
    levels = torch.arange(NUM_THRESHOLDS, device=p_cum.device).unsqueeze(0)
    y_k = (y.unsqueeze(1) > levels).float()
    p = p_cum.clamp(1e-6, 1 - 1e-6)
    return -(y_k * torch.log(p) + (1 - y_k) * torch.log(1 - p)).sum(1)

def shift_fidelity_from_heads(T, S, y):
    """The source shift_fidelity(), applied to cached float32 head outputs instead of a loader.
    T/S: dicts of float32 tensors p_dual/p_macula/p_disc [N,4]; y: long tensor [N]."""
    dt, ds = delta(T["p_dual"], T["p_macula"], T["p_disc"]), delta(S["p_dual"], S["p_macula"], S["p_disc"])
    smae = (ds - dt).abs().sum(1)
    cos = F.cosine_similarity(ds, dt, dim=1, eps=1e-6)
    bt = _nll((T["p_macula"] + T["p_disc"]) / 2, y) - _nll(T["p_dual"], y)
    bs = _nll((S["p_macula"] + S["p_disc"]) / 2, y) - _nll(S["p_dual"], y)
    a, b_ = bt.numpy(), bs.numpy()
    ok = a.std() > 1e-8 and b_.std() > 1e-8
    pear = float(sps.pearsonr(a, b_)[0]) if ok else float("nan")
    spear = float(sps.spearmanr(a, b_)[0]) if ok else float("nan")
    return {"ShiftL1": float(smae.mean()), "ShiftMAE": float(smae.mean() / NUM_THRESHOLDS),
            "CosAgree": float(cos.mean()), "BenefitCorr": pear, "BenefitCorrSpearman": spear}

print("definitions loaded | torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| albumentations", A.__version__)

## 4 — P0 · Environment, inventory, locked configuration, split integrity, checkpoint manifest

In [ ]:
def _ver(pkg):
    try:
        import importlib.metadata as md
        return md.version(pkg)
    except Exception:
        return None

if INFER_DEVICE == "cpu" or (INFER_DEVICE == "auto" and not torch.cuda.is_available()):
    DEVICE = torch.device("cpu")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("INFER_DEVICE='cuda' but no GPU is available")
    DEVICE = torch.device("cuda")
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception as e:
    print("deterministic algorithms unavailable:", e)

ENVIRONMENT = {
    "python": sys.version, "platform": platform.platform(), "torch": torch.__version__,
    "torchvision": torchvision.__version__, "numpy": np.__version__, "pandas": pd.__version__,
    "scipy": _ver("scipy"), "sklearn": _ver("scikit-learn"), "albumentations": A.__version__,
    "opencv": _ver("opencv-python-headless") or _ver("opencv-python"), "matplotlib": _ver("matplotlib"),
    "pyarrow": _ver("pyarrow"), "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "device_used": str(DEVICE),
    "cudnn_allow_tf32": bool(torch.backends.cudnn.allow_tf32),
    "matmul_allow_tf32": bool(torch.backends.cuda.matmul.allow_tf32),
    "in_colab": IN_COLAB, "started_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "orig_art": ORIG_ART, "supp_art": SUPP_ART}
_orig_env_p = f"{O_CFG}/environment.json"
ORIG_ENV = json.load(open(_orig_env_p)) if os.path.exists(_orig_env_p) else {}
ENVIRONMENT["original_run_environment"] = {k: ORIG_ENV.get(k) for k in
                                           ("torch", "torchvision", "albumentations", "gpu", "cuda")}
write_json(ENVIRONMENT, f"{LOGS}/environment.json")
try:
    _fr = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, timeout=180)
    write_text(f"{LOGS}/requirements_supplementary.txt", _fr.stdout)
except Exception as e:
    print("pip freeze unavailable:", e)
gate("G0_Environment", DEVICE.type == "cuda",
     f"device={DEVICE} gpu={ENVIRONMENT['gpu']} torch={torch.__version__} "
     f"(source run: {ENVIRONMENT['original_run_environment']}); CPU is valid, only slower",
     blocking=False)
gate("G0b_AlbumentationsPin", A.__version__ == "1.4.21",
     f"albumentations {A.__version__} (source pinned 1.4.21); resize/normalize must match",
     blocking=False)

# ---- inventory ----
INVENTORY = [("teacher_checkpoint", TEACHER_CKPT, "essential")]
INVENTORY += [(f"student_{c}_s{s}", student_ckpt(c, s), "essential") for c in CORE_CONDITIONS for s in SEEDS_CORE]
INVENTORY += [(f"config_{n}", f"{O_CFG}/{n}.json", "essential") for n in
              ("config", "threshold_calibration", "model_selection", "selected_hyperparameters",
               "split_manifest", "csd_scale")]
INVENTORY += [(f"split_{k}", f"{O_SPLIT}/drtid_{k}.csv", "essential") for k in ("train", "val", "test")]
INVENTORY += [("ref_all_conditions_raw", f"{O_MET}/all_conditions_raw.csv", "reference"),
              ("ref_external_table", f"{O_TAB}/table_external_deepdrid.csv", "reference"),
              ("ref_pred_teacher_test", f"{O_PRED}/test/DRTiD_test_teacher_seed-_FP32.csv", "reference")]
INVENTORY += [(f"ref_pred_{c}_s{s}", f"{O_PRED}/test/DRTiD_test_{c}_seed{s}_FP32.csv", "reference")
              for c in CORE_CONDITIONS for s in SEEDS_CORE]
INVENTORY += [("ref_ext_pred_teacher", f"{O_PRED}/deepdrid/DeepDRiD_setC__1macula_teacher_seed-_FP32.csv", "reference")]
INVENTORY += [(f"ref_ext_pred_csd_s{s}", f"{O_PRED}/deepdrid/DeepDRiD_setC__1macula_csd_fp32_seed{s}_FP32.csv",
               "reference") for s in SEEDS_CORE]
INVENTORY += [(f"ref_confusion_{c}_s{s}", f"{O_MET}/confusion_{c}_seed{s}_FP32.csv", "optional")
              for c in CORE_CONDITIONS for s in SEEDS_CORE]
INVENTORY += [("data_drtid_root", DRTID_ROOT or "<missing>", "essential"),
              ("data_deepdrid_root", DEEPDRID_ROOT or "<missing>", "external")]
_inv_rows = []
for key, p, cat in INVENTORY:
    ex = bool(p) and os.path.exists(p)
    _inv_rows.append({"key": key, "category": cat, "path": p, "exists": ex,
                      "bytes": os.path.getsize(p) if ex and os.path.isfile(p) else None})
INV_DF = pd.DataFrame(_inv_rows)
write_csv(INV_DF, f"{LOGS}/preflight_inventory.csv")
print(INV_DF.groupby(["category", "exists"]).size().to_string())
_miss_ess = INV_DF[(INV_DF.category == "essential") & ~INV_DF.exists]
gate("G1_OriginalArtifactsFound", _miss_ess.empty,
     "all 21 checkpoints, 6 locked configs, 3 splits and DRTiD present" if _miss_ess.empty
     else f"MISSING essential: {_miss_ess.key.tolist()[:8]} -- will NOT retrain")
_miss_ref = INV_DF[(INV_DF.category == "reference") & ~INV_DF.exists]
gate("G1a_ReferenceFilesFound", _miss_ref.empty,
     "all reference outputs of the source run present" if _miss_ref.empty
     else f"missing reference files (fallback to published 4-decimal values where applicable): "
          f"{_miss_ref.key.tolist()[:8]}", blocking=False)
HAVE = dict(zip(INV_DF.key, INV_DF.exists))

# ---- locked configuration must match the source run's own files ----
_cfg = json.load(open(f"{O_CFG}/config.json"))
_thr = json.load(open(f"{O_CFG}/threshold_calibration.json"))["selected_per_condition"]
_sel = json.load(open(f"{O_CFG}/model_selection.json"))
_hp  = json.load(open(f"{O_CFG}/selected_hyperparameters.json"))
_problems = []
if int(_cfg.get("img_size", -1)) != IMG_SIZE: _problems.append(f"img_size {_cfg.get('img_size')}")
if _cfg.get("sampling") != SAMPLING: _problems.append(f"sampling {_cfg.get('sampling')}")
if list(_cfg.get("seeds_core", [])) != SEEDS_CORE: _problems.append(f"seeds {_cfg.get('seeds_core')}")
if tuple(_cfg.get("student_channels", [])) != STUDENT_CHANNELS: _problems.append("student_channels")
if _cfg.get("deepdrid_order") != DEEPDRID_PRIMARY_ORDER: _problems.append("deepdrid_order")
for c, t in LOCKED_THRESHOLDS.items():
    if abs(float(_thr.get(c, -1)) - t) > 1e-12: _problems.append(f"threshold {c}={_thr.get(c)}")
if _sel.get("method") != SELECTED_METHOD: _problems.append(f"M*={_sel.get('method')}")
if _hp.get("logitkd") != LOCKED_HPARAMS["logitkd"] or _hp.get("featkd") != LOCKED_HPARAMS["featkd"] \
        or _hp.get("csd_variant") != LOCKED_HPARAMS["csd_variant"] \
        or abs(float(_hp.get("csd_beta", -1)) - LOCKED_HPARAMS["csd_beta"]) > 1e-12:
    _problems.append(f"hyperparameters {_hp}")
gate("G1b_LockedConfigMatches", not _problems,
     f"384/standard, seeds {SEEDS_CORE}, thresholds {LOCKED_THRESHOLDS}, M*={SELECTED_METHOD}, "
     f"hparams {LOCKED_HPARAMS} -- read from the source run" if not _problems else f"MISMATCH {_problems}")
THRESHOLDS = {c: float(_thr[c]) for c in CORE_CONDITIONS}     # inherited, never tuned

# ---- split integrity ----
_man = json.load(open(f"{O_CFG}/split_manifest.json"))
_sp_rows, _sp_prob = [], []
for k in ("train", "val", "test"):
    p = f"{O_SPLIT}/drtid_{k}.csv"
    raw, lf = sha256_file(p), sha256_lf(p)
    ok = EXPECTED_SPLIT_SHA256[k] in (raw, lf) and _man[k]["sha256"] in (raw, lf)
    _sp_rows.append({"split": k, "sha256_raw": raw, "sha256_lf": lf, "manifest": _man[k]["sha256"],
                     "expected": EXPECTED_SPLIT_SHA256[k], "match": ok})
    if not ok: _sp_prob.append(f"{k} sha mismatch")
TEST_CSV = f"{O_SPLIT}/drtid_test.csv"
TEST_DF = pd.read_csv(TEST_CSV)
if len(TEST_DF) != N_INTERNAL: _sp_prob.append(f"test has {len(TEST_DF)} rows")
if TEST_DF.record_id.duplicated().any(): _sp_prob.append("duplicate record_id")
if not set(TEST_DF.grade.unique()) <= set(range(NUM_CLASSES)): _sp_prob.append("grade outside 0-4")
_missing_img = [f for c in ("macula_filename", "disc_filename") for f in TEST_DF[c]
                if not os.path.exists(f"{DRTID_IMAGE_ROOT}/{f}")]
if _missing_img: _sp_prob.append(f"{len(_missing_img)} test images missing e.g. {_missing_img[:2]}")
write_csv(pd.DataFrame(_sp_rows), f"{LOGS}/split_integrity.csv")
gate("G1c_SplitIntegrity", not _sp_prob,
     f"test split = 550 unique examined-eye records, 1100 images on disk, SHA-256 equals the source "
     f"manifest" if not _sp_prob else " | ".join(_sp_prob))
INTERNAL_FP = EXPECTED_SPLIT_SHA256["test"]

# ---- checkpoint manifest (hash + metadata) ----
def load_raw_ckpt(path):
    return torch.load(path, map_location="cpu", weights_only=False)

def config_fingerprint(cfg):
    return sha256_bytes(json.dumps(cfg, sort_keys=True, default=str).encode()) if isinstance(cfg, dict) else None

EXPECT_CFG = {"dual_no_distill": {"alpha": 0.0, "beta": 0.0, "gamma_feat": 0.0},
              "dual_logitkd": {"alpha": 0.5, "tau_kd": 2.0, "beta": 0.0, "gamma_feat": 0.0},
              "dual_featkd": {"alpha": 0.5, "tau_kd": 2.0, "beta": 0.0, "gamma_feat": 2.0},
              "dual_csd": {"alpha": 0.5, "tau_kd": 2.0, "beta": 0.1, "gamma_feat": 0.0,
                           "csd_variant": "smoothl1_norm", "counterfactual": False}}
CKPT_SHA, _mf_rows, _mf_prob = {}, [], []
for key, path, cond, seed in ([("teacher", TEACHER_CKPT, "teacher", "-")] +
                              [(f"{c}_s{s}", student_ckpt(c, s), c, s) for c in CORE_CONDITIONS for s in SEEDS_CORE]):
    sha = sha256_file(path); CKPT_SHA[key] = sha
    raw = load_raw_ckpt(path)
    cfg = raw.get("config") if isinstance(raw, dict) else None
    row = {"key": key, "condition": cond, "seed": seed, "path": path, "bytes": os.path.getsize(path),
           "sha256": sha, "stored_condition": raw.get("condition") if isinstance(raw, dict) else None,
           "stored_seed": raw.get("seed") if isinstance(raw, dict) else None,
           "best_epoch": raw.get("best_epoch") if isinstance(raw, dict) else None,
           "best_val_qwk": raw.get("best_val_qwk") if isinstance(raw, dict) else None,
           "stored_run_tag": raw.get("run_tag") if isinstance(raw, dict) else None,
           "config_fingerprint": config_fingerprint(cfg)}
    if row["stored_condition"] != cond: _mf_prob.append(f"{key}: stored condition {row['stored_condition']}")
    if cond != "teacher":
        if str(row["stored_seed"]) != str(seed): _mf_prob.append(f"{key}: stored seed {row['stored_seed']}")
        for k, v in EXPECT_CFG[cond].items():
            got = (cfg or {}).get(k)
            if isinstance(v, float) and (got is None or abs(float(got) - v) > 1e-12): _mf_prob.append(f"{key}: {k}={got}")
            elif not isinstance(v, float) and got != v: _mf_prob.append(f"{key}: {k}={got}")
    else:
        if (cfg or {}).get("img_size") != IMG_SIZE or (cfg or {}).get("sampling") != SAMPLING:
            _mf_prob.append(f"teacher recipe {(cfg or {}).get('img_size')}/{(cfg or {}).get('sampling')}")
    if row["stored_run_tag"] not in (None, ORIG_RUN_TAG): _mf_prob.append(f"{key}: run_tag {row['stored_run_tag']}")
    _mf_rows.append(row); del raw
CKPT_MANIFEST = pd.DataFrame(_mf_rows)
write_csv(CKPT_MANIFEST, f"{LOGS}/checkpoint_manifest.csv")
gate("G1d_CheckpointMetadata", not _mf_prob,
     "21 checkpoints hashed; stored condition/seed/config match the locked protocol" if not _mf_prob
     else " | ".join(_mf_prob[:6]))

# Reference files are hashed too, so the end-of-run check proves nothing in the source tree moved.
REF_SHA = {r.key: sha256_file(r.path) for r in INV_DF.itertuples()
           if r.exists and r.category in ("reference", "essential") and os.path.isfile(r.path)
           and not r.key.startswith(("teacher_checkpoint", "student_"))}

# ---- the notebook must not be able to train ----
_FORBIDDEN = ["train_teacher", "train_student", "pretrain_backbone", "run_grid", "prepare_qat",
              "calibrate_threshold", "select_method", "select_checkpoint", "combined_loss", "coral_loss"]
_present = [n for n in _FORBIDDEN if n in globals()]
gate("G1e_NoTrainingCode", not _present and not torch.is_grad_enabled(),
     f"no training/selection function defined; autograd globally disabled={not torch.is_grad_enabled()}"
     if not _present else f"forbidden names present: {_present}")

# Source notebook hash (informational): the repo copy hash is embedded above.
_nb_cands = [p for p in (f"{DRIVE_BASE}/full_pipeline_notebook_enhanced.ipynb",
                         f"{DRIVE_BASE}/notebooks/full_pipeline_notebook_enhanced.ipynb",
                         "/content/drive/MyDrive/Colab Notebooks/full_pipeline_notebook_enhanced.ipynb")
             if os.path.exists(p)]
SOURCE_NB = {"repo_copy_sha256": REPO_SOURCE_NOTEBOOK_SHA256,
             "drive_copy": _nb_cands[0] if _nb_cands else None,
             "drive_copy_sha256": sha256_file(_nb_cands[0]) if _nb_cands else None}
print("source notebook:", SOURCE_NB)

## 5 — Inference engine with a checksummed cache
One forward pass returns all three heads. The teacher is inferred **once per dataset**; each student
once per dataset. A cache entry is reused only if its schema, checkpoint SHA-256, data fingerprint and
file SHA-256 all match — otherwise it is recomputed, never silently mixed.

In [ ]:
def _freeze(m):
    for p in m.parameters(): p.requires_grad_(False)
    return m.eval()

def load_state(path):
    raw = load_raw_ckpt(path)
    return raw["model_state"] if isinstance(raw, dict) and "model_state" in raw else raw

def build_teacher():
    m = Teacher(); m.load_state_dict(load_state(TEACHER_CKPT), strict=True)
    return _freeze(m).to(DEVICE)

def build_student(cond, seed):
    m = Student(); m.load_state_dict(load_state(student_ckpt(cond, seed)), strict=True)
    return _freeze(m).to(DEVICE)

HEADS = ("p_dual", "p_macula", "p_disc")
def head_cols(h): return [f"{h}_{k}" for k in range(NUM_THRESHOLDS)]

@torch.inference_mode()
def infer_three_heads(model, ds, batch_size):
    model.eval()
    ld = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)
    acc = {h: [] for h in HEADS}; y, cid, sid = [], [], []
    for b in ld:
        o = model(b["macula"].to(DEVICE), b["disc"].to(DEVICE))
        for h in HEADS: acc[h].append(o[h].float().cpu())
        y.extend(b["label"].tolist()); cid.extend(b["cluster_id"].tolist()); sid.extend(_ids(b))
    df = pd.DataFrame({"sample_id": sid, "cluster_id": cid, "true_grade": y})
    for h in HEADS:
        arr = torch.cat(acc[h]).numpy().astype(np.float32)
        for k in range(NUM_THRESHOLDS): df[f"{h}_{k}"] = arr[:, k]
    return df

INFER_LOG = []
def cached_heads(dataset_key, model_key, ckpt_sha, builder, ds, batch_size, data_fp):
    base = f"{CACHE}/{dataset_key}/{model_key}"
    meta_p = base + ".meta.json"
    want = {"schema": SCHEMA_VERSION, "dataset": dataset_key, "model_key": model_key,
            "checkpoint_sha256": ckpt_sha, "data_fingerprint": data_fp, "img_size": IMG_SIZE,
            "n_rows": len(ds)}
    if not FORCE_REINFER and os.path.exists(meta_p):
        try:
            meta = json.load(open(meta_p))
            fpath = f"{os.path.dirname(base)}/{meta.get('file')}"
            if all(meta.get(k) == v for k, v in want.items()) and os.path.exists(fpath) \
                    and sha256_file(fpath) == meta.get("file_sha256"):
                df = read_table(base); df["sample_id"] = df["sample_id"].astype(str)
                INFER_LOG.append({**want, "status": "reused", "seconds": 0.0})
                return df
            print(f"  cache for {dataset_key}/{model_key} is stale or incompatible -- recomputing")
        except Exception as e:
            print(f"  cache for {dataset_key}/{model_key} unreadable ({e!r}) -- recomputing")
    t0 = time.time()
    model = builder()
    df = infer_three_heads(model, ds, batch_size)
    del model
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    path = write_table(df, base)
    write_json({**want, "file": os.path.basename(path), "file_sha256": sha256_file(path),
                "device": str(DEVICE), "torch": torch.__version__, "created": time.strftime("%Y-%m-%dT%H:%M:%S")},
               meta_p)
    df["sample_id"] = df["sample_id"].astype(str)
    INFER_LOG.append({**want, "status": "computed", "seconds": round(time.time() - t0, 2)})
    write_csv(pd.DataFrame(INFER_LOG), f"{LOGS}/inference_log.csv")
    return df

def heads_tensors(df):
    return {h: torch.from_numpy(df[head_cols(h)].to_numpy(np.float32).copy()) for h in HEADS}

def heads_np(df):
    # via float32 so a parquet- or CSV-cached table gives bit-identical float64 arrays
    return {h: df[head_cols(h)].to_numpy(np.float32).astype(np.float64) for h in HEADS}

def output_validity(df):
    """finite, inside [0,1], CORAL-monotone (P(y>k) non-increasing in k) for every head."""
    bad = []
    for h in HEADS:
        a = df[head_cols(h)].to_numpy(np.float64)
        if not np.isfinite(a).all(): bad.append(f"{h}: non-finite")
        if (a < 0).any() or (a > 1).any(): bad.append(f"{h}: outside [0,1]")
        v = float((a[:, 1:] - a[:, :-1] > 0).mean())
        if v > 0: bad.append(f"{h}: ordinal violation rate {v:.2e}")
    return bad

print("inference engine ready on", DEVICE)

## 6 — P0 · Smoke reproduction (teacher + CSD seed 42) — STOPS on mismatch
Before any statistics, the frozen teacher and one CSD student must reproduce the source run's saved
per-sample probabilities, grades and per-seed mechanism metrics.

In [ ]:
REF_RAW = None
if HAVE.get("ref_all_conditions_raw"):
    REF_RAW = pd.read_csv(f"{O_MET}/all_conditions_raw.csv", dtype={"seed": str})

def ref_metric(cond, seed, col):
    if REF_RAW is None: return np.nan
    r = REF_RAW[(REF_RAW.condition == cond) & (REF_RAW.seed == str(seed)) & (REF_RAW.quantization == "FP32")]
    return float(r.iloc[0][col]) if len(r) and col in r.columns and pd.notna(r.iloc[0][col]) else np.nan

def saved_pred_path(cond, seed):
    return f"{O_PRED}/test/DRTiD_test_{cond}_seed{seed}_FP32.csv"

def compare_to_saved(df, saved_path, thr, prefix="DRTiD_"):
    """Order, labels, probabilities and grades against the source run's saved prediction file."""
    if not os.path.exists(saved_path):
        return {"saved_available": False}
    sv = pd.read_csv(saved_path, dtype={"sample_id": str})
    mine_ids = [f"{prefix}{s}" for s in df.sample_id]
    same_order = list(sv.sample_id) == mine_ids
    if not same_order:
        return {"saved_available": True, "same_order": False}
    p_mine = df[head_cols("p_dual")].to_numpy(np.float64)
    p_sv = sv[[f"p_gt{k}" for k in range(NUM_THRESHOLDS)]].to_numpy(np.float64)
    g_mine = grade_from(p_mine, thr)
    flip = g_mine != sv.pred_grade.to_numpy()
    near = (np.abs(p_sv[flip] - thr).min(1) <= TOL["flip_margin"]) if flip.any() else np.array([], bool)
    return {"saved_available": True, "same_order": True,
            "labels_equal": bool((sv.true_grade.to_numpy() == df.true_grade.to_numpy()).all()),
            "prob_max_abs_diff": float(np.abs(p_mine - p_sv).max()),
            "prob_mean_abs_diff": float(np.abs(p_mine - p_sv).mean()),
            "grade_agreement": float((~flip).mean()), "n_grade_flips": int(flip.sum()),
            "qwk_saved": float(fast_qwk(sv.true_grade.to_numpy(), sv.pred_grade.to_numpy())),
            "flips_explained": bool(near.all() and flip.mean() <= TOL["flip_max_frac"])}

def saved_ok(c, ext=False):
    if not c.get("saved_available"): return True, "saved predictions unavailable (not compared)"
    if not c.get("same_order"): return False, "sample order differs from the source run"
    ga_min = TOL["ext_grade_agree_min"] if ext else TOL["grade_agree_min"]
    grade_ok = c["grade_agreement"] >= ga_min or c.get("flips_explained") is True or c.get("flips_explained") == True  # noqa: E712
    ok = bool(c["labels_equal"] and c["prob_max_abs_diff"] <= TOL["prob_max"]
              and c["prob_mean_abs_diff"] <= TOL["prob_mean"] and grade_ok)
    return ok, (f"labels_equal={c['labels_equal']} max|dp|={c['prob_max_abs_diff']:.2e} "
                f"mean|dp|={c['prob_mean_abs_diff']:.2e} grade_agree={c['grade_agreement']:.4f} "
                f"(flips={c.get('n_grade_flips', 0)}, all near threshold={c.get('flips_explained', True)})")

def flips_explain_qwk(c):
    """True when the saved file exists and every grade difference is a near-threshold tie (see TOL)."""
    return bool(c.get("saved_available") == True and c.get("same_order") == True  # noqa: E712
                and c.get("flips_explained") == True)  # noqa: E712

TEST_DS = DualViewDataset(TEST_CSV)
prefill_cache(list(TEST_DS.df.macula_path) + list(TEST_DS.df.disc_path), "DRTiD test")

T_INT = cached_heads("internal", "teacher", CKPT_SHA["teacher"], build_teacher, TEST_DS, INTERNAL_BATCH, INTERNAL_FP)
S_INT = {}
S_INT[(CSD, 42)] = cached_heads("internal", f"{CSD}_seed42", CKPT_SHA[f"{CSD}_s42"],
                                lambda: build_student(CSD, 42), TEST_DS, INTERNAL_BATCH, INTERNAL_FP)

_smoke = []
for name, df, thr, cond, seed in (("teacher", T_INT, TEACHER_THRESHOLD, "teacher", "-"),
                                  ("dual_csd_s42", S_INT[(CSD, 42)], THRESHOLDS[CSD], CSD, 42)):
    c = compare_to_saved(df, saved_pred_path(cond, seed), thr)
    ok, det = saved_ok(c)
    bad = output_validity(df)
    q = fast_qwk(df.true_grade, grade_from(df[head_cols("p_dual")].to_numpy(), thr))
    rq = ref_metric(cond, seed, "QWK")
    if not np.isfinite(rq): rq = PAPER_REFERENCE["teacher_QWK"] if cond == "teacher" else np.nan
    _smoke.append({"model": name, "qwk": q, "ref_qwk": rq, **c})
    _qok = (not np.isfinite(rq) or abs(q - rq) <= TOL["qwk_seed"]
            or (flips_explain_qwk(c) and abs(c.get("qwk_saved", np.nan) - rq) <= TOL["qwk_seed"]))
    gate(f"G2_Smoke_{name}", ok and not bad and _qok,
         f"QWK {q:.6f} vs source {rq:.6f}; {det}" + (f"; validity: {bad}" if bad else ""))
_m = shift_fidelity_from_heads(heads_tensors(T_INT), heads_tensors(S_INT[(CSD, 42)]),
                               torch.as_tensor(T_INT.true_grade.to_numpy(), dtype=torch.long))
_dev = {k: abs(_m[k] - ref_metric(CSD, 42, k)) for k in METRICS}
gate("G2_Smoke_MechanismCSDs42",
     all(np.isfinite(v) and v <= TOL["mech_seed"] for v in _dev.values()) if REF_RAW is not None else True,
     ", ".join(f"{k} {_m[k]:.6f} (|d|={_dev[k]:.1e})" for k in METRICS)
     + ("" if REF_RAW is not None else " | per-seed reference unavailable; full check in P1 uses paper means"))
write_csv(pd.DataFrame(_smoke), f"{LOGS}/smoke_reproduction.csv")
PHASE_STATUS["P0"] = "PASSED"
stage_report("P0", "Preflight and smoke reproduction",
             [f"original artifacts: `{ORIG_ART}` (read-only); outputs: `{SUPP_ART}`",
              f"device {DEVICE}; torch {torch.__version__}; albumentations {A.__version__}",
              "21 checkpoints hashed and metadata-checked; locked thresholds/hparams/M* verified",
              f"teacher + CSD seed 42 reproduce the source run (see logs/smoke_reproduction.csv)"])

## 7 — P1 · Inference for all 20 core students on the DRTiD test set + full reproduction gate

In [ ]:
for c in CORE_CONDITIONS:
    for s in SEEDS_CORE:
        if (c, s) not in S_INT:
            S_INT[(c, s)] = cached_heads("internal", f"{c}_seed{s}", CKPT_SHA[f"{c}_s{s}"],
                                         (lambda c=c, s=s: build_student(c, s)), TEST_DS, INTERNAL_BATCH,
                                         INTERNAL_FP)
print("internal inference:", pd.DataFrame(INFER_LOG).groupby("status").size().to_dict())

# ---- record integrity: same 550 records, same order, teacher shared ----
_ids0 = list(T_INT.sample_id)
_int_prob = []
if len(_ids0) != N_INTERNAL or len(set(_ids0)) != N_INTERNAL: _int_prob.append("teacher: not 550 unique records")
if _ids0 != [str(x) for x in TEST_DF.record_id]: _int_prob.append("teacher order != split CSV order")
for k, df in S_INT.items():
    if list(df.sample_id) != _ids0: _int_prob.append(f"{k}: record order differs")
    if not np.array_equal(df.true_grade.to_numpy(), T_INT.true_grade.to_numpy()): _int_prob.append(f"{k}: labels differ")
gate("G3_InternalRecordIntegrity", not _int_prob,
     "21 models x 550 identical records in the split's order; labels identical" if not _int_prob
     else " | ".join(_int_prob[:5]), phase="P1")
_val = {("teacher", "-"): output_validity(T_INT), **{k: output_validity(v) for k, v in S_INT.items()}}
_val = {k: v for k, v in _val.items() if v}
gate("G4_OutputValidity", not _val,
     "all 63 head outputs finite, in [0,1], CORAL-monotone (violation rate 0)" if not _val else str(_val)[:400],
     phase="P1")

# ---- per-model metrics, two independent implementations ----
Y_INT = T_INT.true_grade.to_numpy().astype(int)
Y_INT_T = torch.as_tensor(Y_INT, dtype=torch.long)
TH_INT = heads_tensors(T_INT)

def cosine_np(a, b, eps=1e-6):
    """Independent re-implementation of torch.cosine_similarity (per-norm clamp at eps)."""
    na, nb = np.linalg.norm(a, axis=-1), np.linalg.norm(b, axis=-1)
    return (a * b).sum(-1) / (np.maximum(na, eps) * np.maximum(nb, eps))

def nll_np(p, y):
    yk = (y[:, None] > np.arange(NUM_THRESHOLDS)[None, :]).astype(np.float64)
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return -(yk * np.log(p) + (1 - yk) * np.log(1 - p)).sum(1)

def record_level(Tn, Sn, y):
    """float64 per-record quantities used by every downstream analysis."""
    aggT, aggS = (Tn["p_macula"] + Tn["p_disc"]) / 2, (Sn["p_macula"] + Sn["p_disc"]) / 2
    dT, dS = Tn["p_dual"] - aggT, Sn["p_dual"] - aggS
    l1 = np.abs(dS - dT).sum(1)
    cos = F.cosine_similarity(torch.from_numpy(dS), torch.from_numpy(dT), dim=1, eps=1e-6).numpy()
    bT = nll_np(aggT, y) - nll_np(Tn["p_dual"], y)
    bS = nll_np(aggS, y) - nll_np(Sn["p_dual"], y)
    return {"dT": dT, "dS": dS, "aggT": aggT, "aggS": aggS, "l1": l1, "cos": cos, "bT": bT, "bS": bS,
            "cos_np": cosine_np(dS, dT)}

def pearson_guarded(a, b):
    return float(sps.pearsonr(a, b)[0]) if (a.std() > 1e-8 and b.std() > 1e-8) else float("nan")

def spearman_guarded(a, b):
    return float(sps.spearmanr(a, b)[0]) if (a.std() > 1e-8 and b.std() > 1e-8) else float("nan")

TN_INT = heads_np(T_INT)
REC_INT, ROWS_INT = {}, []
for c in CORE_CONDITIONS:
    for s in SEEDS_CORE:
        df = S_INT[(c, s)]
        rl = record_level(TN_INT, heads_np(df), Y_INT); REC_INT[(c, s)] = rl
        torch_ref = shift_fidelity_from_heads(TH_INT, heads_tensors(df), Y_INT_T)
        pred = grade_from(df[head_cols("p_dual")].to_numpy(), THRESHOLDS[c])
        mets, _ = all_metrics(Y_INT, pred)
        row = {"condition": c, "seed": s, "threshold": THRESHOLDS[c],
               "QWK": mets["QWK"], "QWK_sklearn": float(cohen_kappa_score(Y_INT, pred, weights="quadratic",
                                                                            labels=list(range(NUM_CLASSES)))),
               "MacroF1": mets["MacroF1"], "SevereErrorRate": mets["SevereErrorRate"],
               "ShiftL1": float(rl["l1"].mean()), "ShiftMAE": float(rl["l1"].mean() / NUM_THRESHOLDS),
               "CosAgree": float(rl["cos"].mean()), "BenefitCorr": pearson_guarded(rl["bT"], rl["bS"]),
               "BenefitCorrSpearman": spearman_guarded(rl["bT"], rl["bS"]),
               "CosAgree_numpy_impl": float(rl["cos_np"].mean()),
               **{f"torch_{k}": v for k, v in torch_ref.items()}}
        for k in ("QWK", "ShiftL1", "CosAgree", "BenefitCorr", "BenefitCorrSpearman", "MacroF1", "SevereErrorRate"):
            row[f"ref_{k}"] = ref_metric(c, s, k)
        row["ref_threshold"] = ref_metric(c, s, "decision_threshold")
        row.update({f"saved_{k}": v for k, v in compare_to_saved(df, saved_pred_path(c, s), THRESHOLDS[c]).items()})
        ROWS_INT.append(row)
SEED_MET_INT = pd.DataFrame(ROWS_INT)
_tq = fast_qwk(Y_INT, grade_from(TN_INT["p_dual"], TEACHER_THRESHOLD))
_tsaved = compare_to_saved(T_INT, saved_pred_path("teacher", "-"), TEACHER_THRESHOLD)
TEACHER_INT = {"QWK": _tq, "ref_QWK": ref_metric("teacher", "-", "QWK"), **{f"saved_{k}": v for k, v in _tsaved.items()}}

# ---- G6: reproduction gates ----
REPRO = []
def _add(check, value, reference, tol, kind, explained=False):
    d = abs(value - reference) if (np.isfinite(value) and np.isfinite(reference)) else np.nan
    within = bool(np.isfinite(d) and d <= tol)
    REPRO.append({"check": check, "value": value, "reference": reference, "abs_diff": d, "tolerance": tol,
                  "reference_kind": kind,
                  "explained_by_near_threshold_flips": bool(explained and np.isfinite(d) and not within),
                  "passed": (within or bool(explained and np.isfinite(d))) if np.isfinite(reference) else None})

def _saved_dict(r):
    return {k.replace("saved_", "", 1): getattr(r, k) for k in SEED_MET_INT.columns if k.startswith("saved_")}
# A QWK gap is "explained" only if (i) every grade difference is a near-threshold tie AND (ii) the QWK recomputed
# from the source run's own saved predictions matches the reference -- so a wrong reference can never be excused.
FLIP_OK = {(r.condition, r.seed): flips_explain_qwk(_saved_dict(r)) for r in SEED_MET_INT.itertuples()}
QSAVED = {(r.condition, r.seed): float(_saved_dict(r).get("qwk_saved", np.nan)) for r in SEED_MET_INT.itertuples()}
_tsd = {k.replace("saved_", "", 1): v for k, v in TEACHER_INT.items() if k.startswith("saved_")}
FLIP_OK[("teacher", "-")] = flips_explain_qwk(_tsd)
QSAVED[("teacher", "-")] = float(_tsd.get("qwk_saved", np.nan))
def _expl(keys, saved_value, reference, tol):
    return bool(all(FLIP_OK[k] for k in keys) and np.isfinite(saved_value) and np.isfinite(reference)
                and abs(saved_value - reference) <= tol)
def _keys(conds): return [(c, s) for c in conds for s in SEEDS_CORE]
def _qs_mean(c): return float(np.mean([QSAVED[(c, s)] for s in SEEDS_CORE]))

_tref = TEACHER_INT["ref_QWK"] if np.isfinite(TEACHER_INT["ref_QWK"]) else PAPER_REFERENCE["teacher_QWK"]
_add("teacher_QWK", _tq, _tref, TOL["qwk_seed"] if np.isfinite(TEACHER_INT["ref_QWK"]) else TOL["paper_round"],
     "per-seed source" if np.isfinite(TEACHER_INT["ref_QWK"]) else "paper 4dp",
     explained=_expl([("teacher", "-")], QSAVED[("teacher", "-")], _tref,
                     TOL["qwk_seed"] if np.isfinite(TEACHER_INT["ref_QWK"]) else TOL["paper_round"]))
for r in SEED_MET_INT.itertuples():
    for k, tol in (("QWK", TOL["qwk_seed"]), ("ShiftL1", TOL["mech_seed"]), ("CosAgree", TOL["mech_seed"]),
                   ("BenefitCorr", TOL["mech_seed"]), ("BenefitCorrSpearman", TOL["mech_seed"])):
        _add(f"{r.condition}_s{r.seed}_{k}", getattr(r, k), getattr(r, f"ref_{k}"), tol, "per-seed source",
             explained=(k == "QWK" and _expl([(r.condition, r.seed)], QSAVED[(r.condition, r.seed)],
                                              getattr(r, f"ref_{k}"), tol)))
    if np.isfinite(r.ref_threshold):
        _add(f"{r.condition}_s{r.seed}_threshold", r.threshold, r.ref_threshold, 1e-12, "per-seed source")
MEANS_INT = SEED_MET_INT.groupby("condition")[["QWK", "ShiftL1", "ShiftMAE", "CosAgree", "BenefitCorr",
                                              "BenefitCorrSpearman", "MacroF1", "SevereErrorRate"]].mean()
for c in CORE_CONDITIONS:
    sub = SEED_MET_INT[SEED_MET_INT.condition == c]
    for k in ("QWK", "ShiftL1", "CosAgree", "BenefitCorr"):
        tol = TOL["qwk_mean"] if k == "QWK" else TOL["mech_mean"]
        if sub[f"ref_{k}"].notna().all():
            _add(f"{c}_mean_{k}", MEANS_INT.loc[c, k], sub[f"ref_{k}"].mean(), tol, "5-seed mean, source",
                 explained=(k == "QWK" and _expl(_keys([c]), _qs_mean(c), sub[f"ref_{k}"].mean(), tol)))
        _add(f"{c}_mean_{k}_vs_paper", MEANS_INT.loc[c, k], PAPER_REFERENCE[k][c], TOL["paper_round"], "paper 4dp",
             explained=(k == "QWK" and _expl(_keys([c]), _qs_mean(c), PAPER_REFERENCE[k][c], TOL["paper_round"])))
# predictive RQ1 point differences (published Table II) must also follow from the reproduced seeds
_QWK_BY = {(r.condition, r.seed): r.QWK for r in SEED_MET_INT.itertuples()}
for b, pub in PAPER_REFERENCE["dQWK"].items():
    dq = float(np.mean([_QWK_BY[(CSD, s)] - _QWK_BY[(b, s)] for s in SEEDS_CORE]))
    _add(f"RQ1_dQWK_CSD_vs_{b}_vs_paper", dq, pub, TOL["paper_round"] + 5e-5, "paper Table II",
         explained=_expl(_keys([CSD, b]), _qs_mean(CSD) - _qs_mean(b), pub, TOL["paper_round"] + 5e-5))
REPRO_DF = pd.DataFrame(REPRO)
write_csv(REPRO_DF, f"{LOGS}/reproduction_checks.csv")
write_csv(SEED_MET_INT, f"{TABLES}/table_reproduced_seed_metrics_internal.csv")

_fail = REPRO_DF[REPRO_DF.passed == False]  # noqa: E712
gate("G6_InternalReproduction", _fail.empty,
     f"{int((REPRO_DF.passed == True).sum())} checks within tolerance (max |diff| "  # noqa: E712
     f"{REPRO_DF.abs_diff.max():.2e}); teacher QWK {_tq:.4f}" if _fail.empty
     else f"{len(_fail)} mismatches, e.g. {_fail[['check', 'value', 'reference']].head(4).to_dict('records')}",
     phase="P1")
_sv_bad = []
for r in SEED_MET_INT.itertuples():
    ok, det = saved_ok(_saved_dict(r))
    if not ok: _sv_bad.append(f"{r.condition}_s{r.seed}: {det}")
okT, detT = saved_ok({k.replace("saved_", "", 1): v for k, v in TEACHER_INT.items() if k.startswith("saved_")})
if not okT: _sv_bad.append(f"teacher: {detT}")
_n_saved = int(SEED_MET_INT.get("saved_saved_available", pd.Series(dtype=bool)).fillna(False).sum())
gate("G6a_SavedPredictionAgreement", not _sv_bad,
     f"{_n_saved}/20 student + teacher saved prediction files: order, labels, probabilities and grades agree"
     if not _sv_bad else " | ".join(_sv_bad[:4]), phase="P1")
_impl = max(float((SEED_MET_INT.ShiftL1 - SEED_MET_INT.torch_ShiftL1).abs().max()),
            float((SEED_MET_INT.CosAgree - SEED_MET_INT.torch_CosAgree).abs().max()),
            float((SEED_MET_INT.CosAgree - SEED_MET_INT.CosAgree_numpy_impl).abs().max()),
            float((SEED_MET_INT.BenefitCorr - SEED_MET_INT.torch_BenefitCorr).abs().max()),
            float((SEED_MET_INT.QWK - SEED_MET_INT.QWK_sklearn).abs().max()))
gate("G6b_ImplementationCrossCheck", _impl <= TOL["impl_crosscheck"],
     f"float64 numpy vs source torch shift_fidelity vs independent cosine vs sklearn QWK: max |diff| {_impl:.2e}",
     phase="P1")

# ---- G6c: NLL / benefit spot-check on 10 records with plain-Python math ----
_rng_spot = np.random.default_rng(RNG_SEED)
_spot_idx = _rng_spot.choice(N_INTERNAL, 10, replace=False)
_rl = REC_INT[(CSD, 42)]; _Sn = heads_np(S_INT[(CSD, 42)]); _spot_err = 0.0
for i in _spot_idx:
    def nll1(p, y):
        tot = 0.0
        for k in range(4):
            pk = min(max(float(p[k]), 1e-6), 1 - 1e-6); yk = 1.0 if y > k else 0.0
            tot -= yk * math.log(pk) + (1 - yk) * math.log(1 - pk)
        return tot
    for src, bkey in ((TN_INT, "bT"), (_Sn, "bS")):
        agg = [(src["p_macula"][i][k] + src["p_disc"][i][k]) / 2 for k in range(4)]
        b = nll1(agg, Y_INT[i]) - nll1(src["p_dual"][i], Y_INT[i])
        _spot_err = max(_spot_err, abs(b - _rl[bkey][i]))
gate("G6c_BenefitSpotCheck", _spot_err < 1e-9,
     f"10 random records, teacher and student benefit recomputed in plain Python: max |diff| {_spot_err:.1e}",
     phase="P1")
gate("G5_TeacherCachedOnce", len([r for r in INFER_LOG if r["dataset"] == "internal" and r["model_key"] == "teacher"]) == 1,
     "teacher inferred/loaded exactly once for DRTiD; every student is compared with the same teacher arrays",
     phase="P1")
print(MEANS_INT.round(4).to_string())

## 8 — Eye-level cached table (one row per eye × condition × seed)

In [ ]:
def eye_level_table(dataset, split, order, Tdf, Sdict, rec, thr_map, t_thr, extra_cols):
    rows = []
    Tn = heads_np(Tdf)
    for (c, s), df in Sdict.items():
        rl, Sn = rec[(c, s)], heads_np(df)
        d = pd.DataFrame({"dataset": dataset, "split": split, "field_order": order,
                          "sample_id": Tdf.sample_id.to_numpy(), "patient_id_or_cluster_id": Tdf.cluster_id.to_numpy(),
                          "eye_side": extra_cols["eye_side"], "true_grade": Tdf.true_grade.to_numpy(),
                          "condition": c, "seed": s, "teacher_checkpoint_sha256": CKPT_SHA["teacher"],
                          "student_checkpoint_sha256": CKPT_SHA[f"{c}_s{s}"]})
        for src, tag in ((Tn, "pT"), (Sn, "pS")):
            for h, hn in (("p_dual", "dual"), ("p_macula", "m"), ("p_disc", "d")):
                for k in range(NUM_THRESHOLDS): d[f"{tag}_{hn}_{k}"] = src[h][:, k]
        for k in range(NUM_THRESHOLDS):
            d[f"deltaT_{k}"] = rl["dT"][:, k]; d[f"deltaS_{k}"] = rl["dS"][:, k]
        d["benefitT"], d["benefitS"] = rl["bT"], rl["bS"]
        d["shift_l1_eye"], d["cos_agree_eye"] = rl["l1"], rl["cos"]
        d["pred_grade_T"] = grade_from(Tn["p_dual"], t_thr)
        d["pred_grade_S"] = grade_from(Sn["p_dual"], thr_map[c])
        d["inherited_threshold"] = thr_map[c]
        rows.append(d)
    return pd.concat(rows, ignore_index=True)

_lat = TEST_DF.set_index(TEST_DF.record_id.astype(str)).loc[T_INT.sample_id, "laterality"].astype(str).to_numpy()
EYE_INT = eye_level_table("DRTiD", "test", "macula/disc columns", T_INT, S_INT, REC_INT, THRESHOLDS,
                          TEACHER_THRESHOLD, {"eye_side": _lat})
_p = write_table(EYE_INT, f"{CACHE}/eye_level_internal")
write_json({"schema": SCHEMA_VERSION, "rows": len(EYE_INT), "file": os.path.basename(_p), "sha256": sha256_file(_p),
            "unit": "examined-eye record (DRTiD exposes no patient key)"}, f"{CACHE}/eye_level_internal.meta.json")
gate("G3b_EyeLevelSchema", len(EYE_INT) == 20 * N_INTERNAL and not EYE_INT.duplicated(["sample_id", "condition", "seed"]).any(),
     f"{len(EYE_INT)} rows = 20 models x 550 eyes; key (sample_id, condition, seed) unique", phase="P1")

## 9 — Statistics engine (vectorised, exact)

**Hierarchical paired bootstrap.** Each replicate draws *n* resampling units (DRTiD: examined-eye records;
Set-C: patients, both eyes kept together) and 5 seed indices, all with replacement, and applies **the same draw
to CSD, the baseline and the teacher**. Each condition's metric is recomputed from the resampled records for each
drawn seed (BenefitCorr re-estimated by Pearson inside every replicate) and averaged over the drawn seeds.
A resample is represented by its multiplicity vector, so means and Pearson correlations of the resampled data are
computed exactly as weighted moments; an explicit-loop spot check proves the equivalence on every run.

**Conditional permutation test (secondary).** Mirrors the source notebook's RQ1 permutation: within each
resampling unit the CSD and baseline student outputs are swapped at random, the same swap applied to all five seed
pairs. Null: the two conditions' per-record outputs are exchangeable *given these trained models*. It does not
carry seed-to-seed training variance — which is why the hierarchical CI is the headline and the p-values are
secondary (Holm over the 9 contrasts as one exploratory family).

In [ ]:
class MechData:
    """Arrays for one dataset: teacher shared, students [C,S,N]. Built by sample_id joins."""
    def __init__(self, name, ids, units, y, Tdf, Sdict, conds=CORE_CONDITIONS, seeds=SEEDS_CORE):
        self.name, self.conds, self.seeds = name, list(conds), list(seeds)
        self.ids = np.asarray(ids).astype(str); self.y = np.asarray(y).astype(int)
        Tdf = Tdf.set_index(Tdf.sample_id.astype(str)).loc[self.ids]
        if not np.array_equal(Tdf.true_grade.to_numpy().astype(int), self.y):
            raise SupplementaryBlocked(f"{name}: teacher labels do not match the record list")
        Tn = heads_np(Tdf.reset_index(drop=True))
        C, S, N = len(self.conds), len(self.seeds), len(self.ids)
        self.dT = Tn["p_dual"] - (Tn["p_macula"] + Tn["p_disc"]) / 2
        self.BT = nll_np((Tn["p_macula"] + Tn["p_disc"]) / 2, self.y) - nll_np(Tn["p_dual"], self.y)
        self.dS = np.empty((C, S, N, NUM_THRESHOLDS)); self.BS = np.empty((C, S, N))
        for i, c in enumerate(self.conds):
            for j, s in enumerate(self.seeds):
                d = Sdict[(c, s)]; d = d.set_index(d.sample_id.astype(str)).loc[self.ids]
                if not np.array_equal(d.true_grade.to_numpy().astype(int), self.y):
                    raise SupplementaryBlocked(f"{name}: {c}/{s} labels do not match")
                Sn = heads_np(d.reset_index(drop=True))
                self.dS[i, j] = Sn["p_dual"] - (Sn["p_macula"] + Sn["p_disc"]) / 2
                self.BS[i, j] = nll_np((Sn["p_macula"] + Sn["p_disc"]) / 2, self.y) - nll_np(Sn["p_dual"], self.y)
        self.L1 = np.abs(self.dS - self.dT[None, None]).sum(-1)
        self.COS = F.cosine_similarity(torch.from_numpy(self.dS), torch.from_numpy(
            np.broadcast_to(self.dT, self.dS.shape).copy()), dim=-1, eps=1e-6).numpy()
        self.unit_codes, self.unit_of_eye = np.unique(np.asarray(units), return_inverse=True)
        self.n_units = len(self.unit_codes)
        self.eyes_of_unit = [np.where(self.unit_of_eye == u)[0] for u in range(self.n_units)]
    def ci(self, c): return self.conds.index(c)
    def subset(self, mask, name):
        """Records subset (e.g. teacher-correct eyes); units recomputed from the kept records."""
        out = copy.copy(self); out.name = name
        out.ids, out.y, out.dT, out.BT = self.ids[mask], self.y[mask], self.dT[mask], self.BT[mask]
        out.dS, out.BS, out.L1, out.COS = self.dS[:, :, mask], self.BS[:, :, mask], self.L1[:, :, mask], self.COS[:, :, mask]
        units = self.unit_codes[self.unit_of_eye[mask]]
        out.unit_codes, out.unit_of_eye = np.unique(units, return_inverse=True)
        out.n_units = len(out.unit_codes)
        out.eyes_of_unit = [np.where(out.unit_of_eye == u)[0] for u in range(out.n_units)]
        return out

def seed_metrics(md):
    """Point metrics per (condition, seed) exactly as the source defines them -> dict metric -> [C,S]."""
    C, S = len(md.conds), len(md.seeds)
    out = {"ShiftL1": md.L1.mean(-1), "CosAgree": md.COS.mean(-1),
           "BenefitCorr": np.empty((C, S)), "BenefitCorrSpearman": np.empty((C, S))}
    for i in range(C):
        for j in range(S):
            out["BenefitCorr"][i, j] = pearson_guarded(md.BT, md.BS[i, j])
            out["BenefitCorrSpearman"][i, j] = spearman_guarded(md.BT, md.BS[i, j])
    return out

def _counts(idx, n):
    B = idx.shape[0]
    flat = (idx + np.arange(B)[:, None] * n).ravel()
    return np.bincount(flat, minlength=B * n).reshape(B, n).astype(np.float64)

def make_draws(n_units, n_seeds, B, seed):
    rng = np.random.default_rng(seed)
    r = rng.integers(0, n_units, size=(B, n_units))
    k = rng.integers(0, n_seeds, size=(B, n_seeds))
    return {"B": B, "seed": seed, "r": r, "k": k, "U": _counts(r, n_units), "Sc": _counts(k, n_seeds),
            "hash": sha256_bytes(r.tobytes() + k.tobytes())}

def weighted_pearson(W, x, Y):
    """Pearson of the resampled data, from multiplicities. W [B,N], x [N], Y [K,N] -> [B,K]."""
    x = x - x.mean(); Y = Y - Y.mean(1, keepdims=True)
    n = W.sum(1)[:, None]
    mx = (W @ x)[:, None] / n; my = (W @ Y.T) / n
    vx = (W @ (x * x))[:, None] / n - mx ** 2
    vy = (W @ (Y * Y).T) / n - my ** 2
    cov = (W @ (x[None, :] * Y).T) / n - mx * my
    ok = (vx > 1e-16) & (vy > 1e-16)
    return np.where(ok, cov / np.sqrt(np.where(ok, vx * vy, 1.0)), np.nan)

def weighted_avg_ranks(W, v):
    """Average ranks of the resampled sample (ties -> mean rank, as scipy.rankdata), per replicate."""
    o = np.argsort(v, kind="mergesort"); vs = v[o]
    starts = np.r_[0, np.flatnonzero(np.diff(vs) != 0) + 1]
    grp = np.repeat(np.arange(len(starts)), np.diff(np.r_[starts, len(vs)]))
    Gw = np.add.reduceat(W[:, o], starts, axis=1)
    rank_g = np.cumsum(Gw, 1) - (Gw - 1) / 2
    R = np.empty_like(W); R[:, o] = rank_g[:, grp]
    return R

def weighted_pearson_rows(W, X, Y):
    """Row-wise weighted Pearson where both X and Y vary per replicate. [B,N] each -> [B]."""
    n = W.sum(1)
    mx, my = (W * X).sum(1) / n, (W * Y).sum(1) / n
    Xc, Yc = X - mx[:, None], Y - my[:, None]
    vx, vy = (W * Xc * Xc).sum(1) / n, (W * Yc * Yc).sum(1) / n
    cov = (W * Xc * Yc).sum(1) / n
    ok = (vx > 1e-16) & (vy > 1e-16)
    return np.where(ok, cov / np.sqrt(np.where(ok, vx * vy, 1.0)), np.nan)

def replicate_metrics(md, W, with_spearman=False, chunk=1000):
    """[B,C,S] per metric for every bootstrap replicate (eye weights W)."""
    C, S, N = md.L1.shape
    n = W.sum(1)[:, None]
    out = {"ShiftL1": (W @ md.L1.reshape(C * S, N).T / n).reshape(-1, C, S),
           "CosAgree": (W @ md.COS.reshape(C * S, N).T / n).reshape(-1, C, S),
           "BenefitCorr": weighted_pearson(W, md.BT, md.BS.reshape(C * S, N)).reshape(-1, C, S)}
    if with_spearman:
        sp = np.empty((W.shape[0], C, S))
        for a in range(0, W.shape[0], chunk):
            Wc = W[a:a + chunk]; Rx = weighted_avg_ranks(Wc, md.BT)
            for i in range(C):
                for j in range(S):
                    sp[a:a + chunk, i, j] = weighted_pearson_rows(Wc, Rx, weighted_avg_ranks(Wc, md.BS[i, j]))
        out["BenefitCorrSpearman"] = sp
    return out

def combine_seeds(M, Sc):
    """Seed-resampled condition metric: mean over the drawn seeds (with multiplicity). [B,C,S],[B,S] -> [B,C]."""
    w = Sc[:, None, :]
    use = w > 0
    val = np.where(use, M * w, 0.0).sum(-1) / Sc.sum(1)[:, None]
    val[(np.isnan(M) & use).any(-1)] = np.nan
    return val

def summarise(draws, point):
    v = draws[np.isfinite(draws)]
    if len(v) == 0:
        return {"effect_point": point, "ci_low": np.nan, "ci_high": np.nan, "boot_mean": np.nan,
                "boot_se": np.nan, "n_valid": 0}
    lo, hi = np.percentile(v, [2.5, 97.5])
    return {"effect_point": point, "ci_low": float(lo), "ci_high": float(hi), "boot_mean": float(v.mean()),
            "boot_se": float(v.std(ddof=1)) if len(v) > 1 else np.nan, "n_valid": int(len(v))}

def bootstrap_effects(md, draws, metrics=METRICS, seed_resample=True, with_spearman=False):
    """{(baseline, metric): (effect draws [B], point effect)} for CSD minus each baseline."""
    W = draws["U"][:, md.unit_of_eye]
    R = replicate_metrics(md, W, with_spearman=with_spearman)
    Sc = draws["Sc"] if seed_resample else np.ones_like(draws["Sc"])
    P = seed_metrics(md)
    res = {}
    for m in metrics:
        Mc = combine_seeds(R[m], Sc)
        for b in BASELINES:
            e = Mc[:, md.ci(CSD)] - Mc[:, md.ci(b)]
            point = float(np.mean(P[m][md.ci(CSD)] - P[m][md.ci(b)]))
            res[(b, m)] = (e, point)
    return res, R, P

def contrast_draws(md, draws, a, b, metric):
    """Effect draws for an arbitrary pair (used by the sanity checks)."""
    W = draws["U"][:, md.unit_of_eye]
    Mc = combine_seeds(replicate_metrics(md, W)[metric], draws["Sc"])
    return Mc[:, md.ci(a)] - Mc[:, md.ci(b)]

def permutation_test(md, a, b, metric, P, seed):
    """Conditional paired permutation (swap within resampling unit, same swap for every seed pair)."""
    rng = np.random.default_rng(seed)
    sw = (rng.random((P, md.n_units)) < 0.5).astype(np.float64)[:, md.unit_of_eye]     # [P,N]
    ia, ib, N = md.ci(a), md.ci(b), len(md.ids)
    if metric in ("ShiftL1", "CosAgree"):
        V = md.L1 if metric == "ShiftL1" else md.COS
        D = (V[ia] - V[ib]).mean(0)                                                      # [N]
        obs = float(D.mean())
        perm = ((1.0 - 2.0 * sw) @ D) / N
    else:
        x = md.BT - md.BT.mean(); Sx, Sxx = x.sum(), (x * x).sum()
        def r_from(Sy, Syy, Sxy):
            vx = Sxx / N - (Sx / N) ** 2; vy = Syy / N - (Sy / N) ** 2
            cov = Sxy / N - (Sx / N) * (Sy / N)
            ok = (vy > 1e-16) & (vx > 1e-16)
            return np.where(ok, cov / np.sqrt(np.where(ok, vx * vy, 1.0)), np.nan)
        perm = np.zeros(P); obs = 0.0
        for j in range(len(md.seeds)):
            ya, yb = md.BS[ia, j].copy(), md.BS[ib, j].copy()
            c0 = 0.5 * (ya.mean() + yb.mean()); ya -= c0; yb -= c0
            d, d2, dx = yb - ya, yb * yb - ya * ya, x * (yb - ya)
            ra = r_from(ya.sum() + sw @ d, (ya * ya).sum() + sw @ d2, (x * ya).sum() + sw @ dx)
            rb = r_from(yb.sum() - sw @ d, (yb * yb).sum() - sw @ d2, (x * yb).sum() - sw @ dx)
            perm += (ra - rb) / len(md.seeds)
            obs += (pearson_guarded(md.BT, md.BS[ia, j]) - pearson_guarded(md.BT, md.BS[ib, j])) / len(md.seeds)
    ok = np.isfinite(perm)
    cnt = int((np.abs(perm[ok]) >= abs(obs) - 1e-12).sum())
    return {"observed": obs, "p_perm": (cnt + 1) / (int(ok.sum()) + 1), "n_perm_valid": int(ok.sum())}

def holm(pvals, names, alpha=0.05):
    m = len(pvals); order = np.argsort(pvals); adj = [None] * m; run = 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, (m - rank) * pvals[i])); adj[i] = run
    return {names[i]: {"p_raw": float(pvals[i]), "p_holm": float(adj[i])} for i in range(m)}

def explicit_replicate(md, draws, b_idx, cond, metric):
    """Ground truth for the spot check: literally resample, recompute, average over drawn seeds."""
    eyes = np.concatenate([md.eyes_of_unit[u] for u in draws["r"][b_idx]])
    vals = []
    for j in draws["k"][b_idx]:
        i = md.ci(cond)
        if metric == "ShiftL1": vals.append(md.L1[i, j, eyes].mean())
        elif metric == "CosAgree": vals.append(md.COS[i, j, eyes].mean())
        elif metric == "BenefitCorr": vals.append(pearson_guarded(md.BT[eyes], md.BS[i, j, eyes]))
        else: vals.append(spearman_guarded(md.BT[eyes], md.BS[i, j, eyes]))
    return float(np.mean(vals))

def bootstrap_sanity(md, phase, seed):
    """Gate 8: zero-difference, order reversal, explicit-loop equivalence, RNG reproducibility."""
    d = make_draws(md.n_units, len(md.seeds), 300, seed)
    checks = []
    for m in METRICS:
        z = contrast_draws(md, d, CSD, CSD, m)
        checks.append((f"identical_{m}", float(np.nanmax(np.abs(z))) == 0.0, f"max|effect|={np.nanmax(np.abs(z)):.1e}"))
        e1, e2 = contrast_draws(md, d, CSD, BASELINES[0], m), contrast_draws(md, d, BASELINES[0], CSD, m)
        s1, s2 = summarise(e1, 0.0), summarise(e2, 0.0)
        checks.append((f"reversal_{m}", np.allclose(e1, -e2, equal_nan=True) and
                       abs(s1["ci_low"] + s2["ci_high"]) < 1e-12 and abs(s1["ci_high"] + s2["ci_low"]) < 1e-12,
                       f"CI {s1['ci_low']:.4f},{s1['ci_high']:.4f} vs reversed {s2['ci_low']:.4f},{s2['ci_high']:.4f}"))
    W = d["U"][:, md.unit_of_eye]
    R = replicate_metrics(md, W, with_spearman=True)
    worst = 0.0
    for b_idx in range(20):
        for m in METRICS + ["BenefitCorrSpearman"]:
            for cond in (CSD, BASELINES[0]):
                vec = combine_seeds(R[m][b_idx:b_idx + 1], d["Sc"][b_idx:b_idx + 1])[0, md.ci(cond)]
                worst = max(worst, abs(vec - explicit_replicate(md, d, b_idx, cond, m)))
    checks.append(("explicit_loop_equivalence", worst < 1e-9, f"20 replicates x 4 metrics x 2 conditions: max |diff| {worst:.1e}"))
    d2 = make_draws(md.n_units, len(md.seeds), 300, seed)
    checks.append(("rng_reproducible", d2["hash"] == d["hash"] and
                   np.array_equal(contrast_draws(md, d2, CSD, BASELINES[1], "BenefitCorr"),
                                  contrast_draws(md, d, CSD, BASELINES[1], "BenefitCorr"), equal_nan=True),
                   f"draw hash {d['hash'][:12]}"))
    pz = permutation_test(md, CSD, CSD, "CosAgree", 200, seed)
    checks.append(("permutation_identical", pz["observed"] == 0.0 and pz["p_perm"] == 1.0, f"p={pz['p_perm']}"))
    # cluster integrity: every eye of a unit receives the same weight in every replicate
    ci_ok = all(np.all(W[:, e] == W[:, e[:1]]) for e in md.eyes_of_unit)
    checks.append(("cluster_weights_shared", ci_ok, f"{md.n_units} units never split"))
    df = pd.DataFrame([{"phase": phase, "dataset": md.name, "check": c, "passed": bool(p), "detail": t}
                       for c, p, t in checks])
    return df

print("statistics engine ready")

## 10 — P1 · Internal mechanism inference: 9 exploratory contrasts (CSD − baseline)

In [ ]:
MD_INT = MechData("DRTiD_test", T_INT.sample_id, T_INT.cluster_id, Y_INT, T_INT, S_INT)

# shuffled-order invariance: rebuild from shuffled caches -> identical arrays (pairing is by sample_id)
_rng_sh = np.random.default_rng(RNG_SEED + 1)
_sh = MechData("shuffled", T_INT.sample_id, T_INT.cluster_id, Y_INT,
               T_INT.sample(frac=1.0, random_state=1).reset_index(drop=True),
               {k: v.sample(frac=1.0, random_state=int(_rng_sh.integers(1e9))).reset_index(drop=True)
                for k, v in S_INT.items()})
_sh_ok = all(np.array_equal(getattr(_sh, a), getattr(MD_INT, a)) for a in ("dT", "BT", "dS", "BS", "L1", "COS"))
del _sh
SANITY_INT = bootstrap_sanity(MD_INT, "P1", RNG_SEED + 2)
SANITY_INT = pd.concat([SANITY_INT, pd.DataFrame([{
    "phase": "P1", "dataset": MD_INT.name, "check": "shuffled_order_invariance", "passed": bool(_sh_ok),
    "detail": "ID-joined arrays identical after shuffling every cache"}])], ignore_index=True)
write_csv(SANITY_INT, f"{LOGS}/bootstrap_sanity_internal.csv")
print(SANITY_INT.to_string(index=False))
gate("G8_BootstrapSanity_Internal", SANITY_INT.passed.all(),
     f"{int(SANITY_INT.passed.sum())}/{len(SANITY_INT)} sanity checks" if SANITY_INT.passed.all()
     else f"failed: {SANITY_INT[~SANITY_INT.passed].check.tolist()}", phase="P1")
_pt = seed_metrics(MD_INT)
_pt_dev = max(float(np.nanmax(np.abs(_pt[m] - SEED_MET_INT.pivot(index="condition", columns="seed", values=m)
                                      .loc[CORE_CONDITIONS, SEEDS_CORE].to_numpy()))) for m in METRICS)
gate("G7_BootstrapPairing", _pt_dev < 1e-12,
     f"statistics arrays (built by sample_id joins) reproduce the per-seed table exactly (max |diff| {_pt_dev:.1e}); "
     "one draw object is shared by CSD, baseline and teacher in every replicate", phase="P1")

# ---- pilot (pipeline + timing), then FINAL ----
t0 = time.time()
_pilot, _, _ = bootstrap_effects(MD_INT, make_draws(MD_INT.n_units, 5, B_PILOT, RNG_SEED + 10))
_pilot_df = pd.DataFrame([{"baseline": b, "metric": m, **summarise(e, p)} for (b, m), (e, p) in _pilot.items()])
write_csv(_pilot_df, f"{LOGS}/pilot/pilot_internal_B{B_PILOT}.csv")
print(f"pilot B={B_PILOT}: {time.time() - t0:.1f}s (pilot intervals are NOT reported)")

def mechanism_stats(md, draws_seed, perm_seed, phase, dataset_label, unit_label):
    t0 = time.time()
    DR = make_draws(md.n_units, len(md.seeds), B_FINAL, draws_seed)
    hier, R, P = bootstrap_effects(md, DR, with_spearman=True, metrics=METRICS + ["BenefitCorrSpearman"])
    rec, _, _ = bootstrap_effects(md, DR, seed_resample=False)
    perms = {(b, m): permutation_test(md, CSD, b, m, P_PERM, perm_seed + 100 * i + j)
             for i, b in enumerate(BASELINES) for j, m in enumerate(METRICS)}
    names = [f"{b}|{m}" for b in BASELINES for m in METRICS]
    H = holm([perms[(b, m)]["p_perm"] for b in BASELINES for m in METRICS], names)
    rows, seed_rows, spear_rows = [], [], []
    for b in BASELINES:
        for m in METRICS:
            e, point = hier[(b, m)]
            s = summarise(e, point)
            sr = summarise(rec[(b, m)][0], point)
            per_seed = P[m][md.ci(CSD)] - P[m][md.ci(b)]
            fav = (per_seed < 0) if BETTER[m] == "lower" else (per_seed > 0)
            loso = [float(np.mean(np.delete(per_seed, j))) for j in range(len(md.seeds))]
            excl = bool(s["ci_low"] > 0 or s["ci_high"] < 0)
            good_dir = (point < 0) if BETTER[m] == "lower" else (point > 0)
            if not excl: interp = "CI includes 0 -- not conclusive"
            elif good_dir: interp = "CI excludes 0 -- favours CSD"
            else: interp = "CI excludes 0 -- favours baseline"
            pr = perms[(b, m)]
            rows.append({"dataset": dataset_label, "contrast": f"CSD - {LABEL[b]}", "baseline": b, "metric": m,
                         "better": BETTER[m], "csd_mean": float(P[m][md.ci(CSD)].mean()),
                         "baseline_mean": float(P[m][md.ci(b)].mean()), **s,
                         "ci_excludes_zero": excl, "point_favours_csd": bool(good_dir),
                         "conclusive_favours_csd": bool(excl and good_dir), "interpretation": interp,
                         "B": B_FINAL, "recordonly_ci_low": sr["ci_low"], "recordonly_ci_high": sr["ci_high"],
                         "recordonly_excludes_zero": bool(sr["ci_low"] > 0 or sr["ci_high"] < 0),
                         "p_perm_conditional": pr["p_perm"], "p_holm_9": H[f"{b}|{m}"]["p_holm"],
                         "n_perm": pr["n_perm_valid"], "seeds_favouring_csd": int(fav.sum()),
                         "n_seeds": len(md.seeds), "loso_min": min(loso), "loso_max": max(loso),
                         "loso_sign_stable": bool(all(np.sign(x) == np.sign(point) for x in loso)),
                         "n_records": len(md.ids), "n_units": md.n_units, "unit": unit_label,
                         "draw_hash": DR["hash"][:16],
                         "analysis": "hierarchical paired bootstrap (units + matched seeds), exploratory post-hoc"})
            for j, sd in enumerate(md.seeds):
                seed_rows.append({"dataset": dataset_label, "contrast": f"CSD - {LABEL[b]}", "metric": m, "seed": sd,
                                  "csd_value": float(P[m][md.ci(CSD), j]), "baseline_value": float(P[m][md.ci(b), j]),
                                  "effect": float(per_seed[j]), "favours_csd": bool(fav[j]),
                                  "loso_effect_without_this_seed": loso[j]})
        e, point = hier[(b, "BenefitCorrSpearman")]
        s = summarise(e, point)
        spear_rows.append({"dataset": dataset_label, "contrast": f"CSD - {LABEL[b]}", "metric": "BenefitCorrSpearman",
                           "csd_mean": float(P["BenefitCorrSpearman"][md.ci(CSD)].mean()),
                           "baseline_mean": float(P["BenefitCorrSpearman"][md.ci(b)].mean()), **s,
                           "ci_excludes_zero": bool(s["ci_low"] > 0 or s["ci_high"] < 0), "B": B_FINAL,
                           "role": "sensitivity only (not part of the 9-contrast family)"})
    print(f"{phase}: B={B_FINAL} hierarchical + record-only bootstrap, {P_PERM} permutations x 9 "
          f"in {time.time() - t0:.1f}s")
    return pd.DataFrame(rows), pd.DataFrame(seed_rows), pd.DataFrame(spear_rows), P, DR

STATS_INT, SEEDCONS_INT, SPEAR_INT, P_INT, DR_INT = mechanism_stats(
    MD_INT, RNG_SEED + 20, RNG_SEED + 30, "P1", "DRTiD internal test", "examined-eye record")
_nvalid_min = int(STATS_INT.n_valid.min())
gate("G9_DegenerateReplicates", _nvalid_min == B_FINAL,
     f"valid replicates per contrast: min {_nvalid_min}/{B_FINAL} (Pearson undefined replicates are counted, "
     "never silently dropped)", blocking=False, phase="P1")
write_csv(STATS_INT, f"{TABLES}/table_mechanism_internal_stats.csv")
write_csv(SEEDCONS_INT, f"{TABLES}/table_mechanism_seed_consistency.csv")
write_csv(SPEAR_INT, f"{TABLES}/table_mechanism_internal_spearman_sensitivity.csv")
# point estimates must equal the reproduced 5-seed means exactly
_pe = max(abs(r.effect_point - (MEANS_INT.loc[CSD, r.metric] - MEANS_INT.loc[r.baseline, r.metric]))
          for r in STATS_INT.itertuples())
gate("G7b_PointEffectsConsistent", _pe < 1e-12,
     f"point effects = differences of reproduced 5-seed means (max |diff| {_pe:.1e})", phase="P1")
_show = ["contrast", "metric", "effect_point", "ci_low", "ci_high", "interpretation", "seeds_favouring_csd",
         "recordonly_excludes_zero", "p_perm_conditional", "p_holm_9"]
print(STATS_INT[_show].round(4).to_string(index=False))
PHASE_STATUS["P1"] = "PASSED"

### Forest figure (internal)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 8, "axes.titlesize": 8.5, "axes.labelsize": 8, "xtick.labelsize": 7.5,
                     "ytick.labelsize": 8, "legend.fontsize": 7.5, "svg.fonttype": "none",
                     "pdf.fonttype": 42, "font.family": "DejaVu Sans"})
INK, MUTED, INK2 = "#1f3b57", "#8a96a3", "#6b8fb3"

def save_fig(fig, stem, caption, data):
    for ext in ("png", "svg", "pdf"):
        _atomic(f"{FIGS}/{stem}.{ext}", lambda p, ext=ext: fig.savefig(p, format=ext, dpi=300, bbox_inches="tight"))
    write_text(f"{FIGS}/{stem}_caption.txt", caption)
    write_csv(data, f"{FIGS}/{stem}_data.csv")
    plt.close(fig)

def _xlabel(m):
    arrow = "← favours CSD" if BETTER[m] == "lower" else "favours CSD →"
    return f"Δ{m} (CSD − baseline)\n{arrow}"

def forest(stats_list, labels, markers, stem, caption, title_suffix=""):
    fig, axes = plt.subplots(1, 3, figsize=(7.1, 2.25), sharey=True)
    yb = {b: i for i, b in enumerate(reversed(BASELINES))}
    off = np.linspace(-0.14, 0.14, len(stats_list)) if len(stats_list) > 1 else [0.0]
    for ax, m in zip(axes, METRICS):
        ax.axvline(0, color=MUTED, lw=0.8, ls="--", zorder=0)
        for k, (st, lab, mk, o) in enumerate(zip(stats_list, labels, markers, off)):
            col = INK if k == 0 else INK2
            d = st[st.metric == m]
            for r in d.itertuples():
                y = yb[r.baseline] - o          # first series drawn on top
                ax.plot([r.ci_low, r.ci_high], [y, y], color=col, lw=1.2, solid_capstyle="butt")
                ax.plot(r.effect_point, y, marker=mk, ms=5, mec=col, mfc=col if r.ci_excludes_zero else "white",
                        mew=1.1, ls="none", label=lab)
        ax.set_title(m + title_suffix)
        ax.set_xlabel(_xlabel(m))
        ax.grid(axis="x", color="#e3e7eb", lw=0.6)
        for s in ("top", "right"): ax.spines[s].set_visible(False)
    axes[0].set_yticks(list(yb.values()))
    axes[0].set_yticklabels([f"vs {LABEL[b]}" for b in yb])
    if len(stats_list) > 1:
        h, l = axes[0].get_legend_handles_labels()
        uniq = dict(zip(l, h))
        fig.legend(uniq.values(), uniq.keys(), loc="upper center", ncol=len(uniq), frameon=False,
                   bbox_to_anchor=(0.5, 1.08))
    fig.tight_layout()
    data = pd.concat([s.assign(series=lab) for s, lab in zip(stats_list, labels)], ignore_index=True)
    save_fig(fig, stem, caption, data[["series", "contrast", "metric", "effect_point", "ci_low", "ci_high",
                                       "ci_excludes_zero", "B", "n_units", "unit"]])

B_TXT = f"{B_FINAL:,}".replace(",", ".")
CAP_FOREST_INT = (
    "Gambar suplementer. Selisih fidelitas mekanisme CSD terhadap tiga pembanding pada 550 mata data uji internal "
    f"DRTiD (rerata lima seed berpasangan; titik = estimasi, garis = CI 95% hierarchical paired bootstrap, B = {B_TXT}"
    "). Titik terisi: CI tidak memuat nol; titik kosong: CI memuat nol. ShiftL1 lebih rendah lebih baik "
    "(arah ke kiri menguntungkan CSD); CosAgree dan BenefitCorr lebih tinggi lebih baik. Analisis eksploratori pasca-hoc; "
    "unit resampling adalah examined-eye record.\n\n"
    "Supplementary figure. Mechanism-fidelity effects of CSD versus three baselines on the 550-eye DRTiD internal test "
    f"set (mean of five matched seeds; whiskers = 95% hierarchical paired bootstrap CI, B = {B_FINAL}). Filled marker: CI "
    "excludes zero. Lower ShiftL1 and higher CosAgree/BenefitCorr favour CSD. Exploratory post-hoc analysis.")
forest([STATS_INT], ["DRTiD internal"], ["o"], "fig_mechanism_effect_forest", CAP_FOREST_INT)
stage_report("P1", "Internal reproduction and mechanism inference",
             [f"teacher QWK {_tq:.4f}; 5-seed means reproduced (max |diff| {REPRO_DF.abs_diff.max():.1e})",
              f"9 contrasts, hierarchical paired bootstrap B={B_FINAL}; "
              f"{int(STATS_INT.conclusive_favours_csd.sum())}/9 CIs exclude 0 in CSD's favour, "
              f"{int((STATS_INT.ci_excludes_zero & ~STATS_INT.point_favours_csd).sum())}/9 in a baseline's favour",
              "sanity: identical-method zero, order reversal, explicit-loop equivalence, RNG reproducibility, "
              "shuffled-order invariance all passed"])

## 11 — P2 · External mechanism generalisation on DeepDRiD Set-C (post-hoc, exploratory)
Same unchanged FP32 models and thresholds, primary field order `_1=macula`, **patient-clustered** paired bootstrap
(both eyes of a resampled patient stay together). This comparison was designed after the main external results were
known, so it is exploratory. The published external CSD QWK (0.6688) must be reproduced first; no new external winner
is ever selected.

In [ ]:
STATS_EXT = SEEDCONS_EXT = SPEAR_EXT = MD_EXT = None
EXT_QWK_TABLE = EXT_QWK_PAIRED = None
P2_BLOCKER = None
remove_stale([f"{TABLES}/{f}" for f in (
    "table_mechanism_external_stats.csv", "table_mechanism_seed_consistency_external.csv",
    "table_mechanism_external_spearman_sensitivity.csv", "table_external_core_qwk_exploratory.csv",
    "table_external_core_qwk_paired_exploratory.csv", "table_mechanism_rank_internal_vs_external.csv",
    "table_teacher_shift_profile.csv", "table_VI_paper_external_optional.csv")]
             + glob.glob(f"{FIGS}/fig_internal_external_mechanism*"))
if not RUN_EXTERNAL:
    PHASE_STATUS["P2"] = "SKIPPED_BY_CONFIG"
elif PHASE_STATUS["P1"] != "PASSED":
    PHASE_STATUS["P2"] = "NOT_RUN_P1_FAILED"
else:
    try:
        if DEEPDRID_ROOT is None:
            gate("G10_SetCAvailable", False, "DeepDRiD not found under dataset/", phase="P2")
        SETC = SetCDataset(DEEPDRID_ROOT)
        _npat = int(SETC.df.patient_id.nunique()) if len(SETC.df) else 0
        _eyes_pp = SETC.df.groupby("patient_id").eye.nunique() if len(SETC.df) else pd.Series(dtype=int)
        gate("G10_SetCIntegrity", SETC.available and len(SETC.df) == N_EXT_EYES and _npat == N_EXT_PATIENTS
             and len(SETC.excl) == 0 and bool((_eyes_pp == 2).all())
             and set(SETC.df.grade.unique()) <= set(range(NUM_CLASSES)),
             f"{_npat} patients / {len(SETC.df)} eyes / {2 * len(SETC.df)} images, {len(SETC.excl)} exclusions, "
             f"every patient has both eyes; order {SETC.order}", phase="P2")
        EXT_FP = sha256_bytes(json.dumps(
            [[int(r.patient_id), r.eye, int(r.grade), os.path.basename(r.f1), os.path.basename(r.f2)]
             for r in SETC.df.itertuples()]).encode())
        prefill_cache(list(SETC.df.f1) + list(SETC.df.f2), "Set-C")
        T_EXT = cached_heads("external", "teacher", CKPT_SHA["teacher"], build_teacher, SETC, EXTERNAL_BATCH, EXT_FP)
        S_EXT = {(c, s): cached_heads("external", f"{c}_seed{s}", CKPT_SHA[f"{c}_s{s}"],
                                      (lambda c=c, s=s: build_student(c, s)), SETC, EXTERNAL_BATCH, EXT_FP)
                 for c in CORE_CONDITIONS for s in SEEDS_CORE}
        _xbad = [k for k, v in S_EXT.items() if list(v.sample_id) != list(T_EXT.sample_id)]
        _xval = {k: output_validity(v) for k, v in {**S_EXT, ("teacher", "-"): T_EXT}.items()}
        _xval = {k: v for k, v in _xval.items() if v}
        gate("G10b_ExternalOutputs", not _xbad and not _xval and len(T_EXT) == N_EXT_EYES,
             "21 models x 200 eyes, same order; all heads finite, in [0,1], CORAL-monotone"
             if not (_xbad or _xval) else f"order {_xbad[:3]} validity {str(_xval)[:200]}", phase="P2")

        # ---- reproduce the published external teacher / CSD results before anything new ----
        Y_EXT = T_EXT.true_grade.to_numpy().astype(int)
        def ext_saved(cond, seed): return f"{O_PRED}/deepdrid/DeepDRiD_setC__1macula_{cond}_seed{seed}_FP32.csv"
        _ref_ext = None
        if HAVE.get("ref_external_table"):
            _e = pd.read_csv(f"{O_TAB}/table_external_deepdrid.csv", dtype={"seed": str})
            _ref_ext = _e[(_e.subset == "setC") & (_e.field_order == DEEPDRID_PRIMARY_ORDER)]
        def ref_ext_qwk(cond, seed):
            if _ref_ext is None: return np.nan
            r = _ref_ext[(_ref_ext.condition == cond) & (_ref_ext.seed == str(seed))]
            return float(r.iloc[0].QWK) if len(r) else np.nan
        XREP, _xprob = [], []
        for cond, seed, df, thr in ([("teacher", "-", T_EXT, TEACHER_THRESHOLD)] +
                                    [("csd_fp32", s, S_EXT[(CSD, s)], THRESHOLDS[CSD]) for s in SEEDS_CORE]):
            q = fast_qwk(Y_EXT, grade_from(df[head_cols("p_dual")].to_numpy(), thr))
            rq = ref_ext_qwk(cond, seed)
            c = compare_to_saved(df, ext_saved(cond, seed), thr, prefix="DeepDRiD_")
            ok, det = saved_ok(c, ext=True)
            XREP.append({"model": cond, "seed": seed, "QWK": q, "ref_QWK": rq,
                         "abs_diff": abs(q - rq) if np.isfinite(rq) else np.nan, **c})
            if not ok: _xprob.append(f"{cond}/{seed}: {det}")
            _xe = flips_explain_qwk(c) and np.isfinite(rq) and abs(c.get("qwk_saved", np.nan) - rq) <= TOL["ext_qwk_seed"]
            if np.isfinite(rq) and abs(q - rq) > TOL["ext_qwk_seed"] and not _xe:
                _xprob.append(f"{cond}/{seed}: QWK {q:.4f} vs {rq:.4f}")
            XREP[-1]["flips_explain_qwk"] = bool(flips_explain_qwk(c))
        XREP = pd.DataFrame(XREP)
        _x_expl = bool(XREP.flips_explain_qwk.all()) and "qwk_saved" in XREP.columns
        _qs_t = float(XREP[XREP.model == "teacher"].qwk_saved.iloc[0]) if _x_expl else np.nan
        _qs_c = float(XREP[XREP.model == "csd_fp32"].qwk_saved.mean()) if _x_expl else np.nan
        _csd_mean = float(XREP[XREP.model == "csd_fp32"].QWK.mean())
        _t_ext = float(XREP[XREP.model == "teacher"].QWK.iloc[0])
        if abs(_csd_mean - PAPER_REFERENCE["ext_csd_QWK"]) > TOL["ext_qwk_mean"] + 5e-5 and not (
                _x_expl and abs(_qs_c - PAPER_REFERENCE["ext_csd_QWK"]) <= TOL["ext_qwk_mean"] + 5e-5):
            _xprob.append(f"CSD external mean QWK {_csd_mean:.4f} vs published {PAPER_REFERENCE['ext_csd_QWK']}")
        if abs(_t_ext - PAPER_REFERENCE["ext_teacher_QWK"]) > TOL["ext_qwk_mean"] + 5e-5 and not (
                _x_expl and abs(_qs_t - PAPER_REFERENCE["ext_teacher_QWK"]) <= TOL["ext_qwk_mean"] + 5e-5):
            _xprob.append(f"teacher external QWK {_t_ext:.4f} vs published {PAPER_REFERENCE['ext_teacher_QWK']}")
        write_csv(XREP, f"{LOGS}/reproduction_checks_external.csv")
        gate("G11_ExternalReproduction", not _xprob,
             f"teacher QWK {_t_ext:.4f} (pub. {PAPER_REFERENCE['ext_teacher_QWK']}); CSD 5-seed mean {_csd_mean:.4f} "
             f"(pub. {PAPER_REFERENCE['ext_csd_QWK']}); "
             f"saved predictions agree" if not _xprob else " | ".join(_xprob[:4]), phase="P2")

        # ---- external mechanism statistics (patient-clustered) ----
        TN_EXT = heads_np(T_EXT)
        REC_EXT = {(c, s): record_level(TN_EXT, heads_np(S_EXT[(c, s)]), Y_EXT) for c in CORE_CONDITIONS for s in SEEDS_CORE}
        EYE_EXT = eye_level_table("DeepDRiD", "setC", DEEPDRID_PRIMARY_ORDER, T_EXT, S_EXT, REC_EXT, THRESHOLDS,
                                  TEACHER_THRESHOLD, {"eye_side": T_EXT.sample_id.str.split("_").str[1].to_numpy()})
        _p = write_table(EYE_EXT, f"{CACHE}/eye_level_external")
        write_json({"schema": SCHEMA_VERSION, "rows": len(EYE_EXT), "file": os.path.basename(_p),
                    "sha256": sha256_file(_p), "unit": "eye; cluster = patient"}, f"{CACHE}/eye_level_external.meta.json")
        gate("G10c_EyeLevelSchemaExternal", len(EYE_EXT) == 20 * N_EXT_EYES
             and not EYE_EXT.duplicated(["sample_id", "condition", "seed"]).any(),
             f"{len(EYE_EXT)} rows = 20 models x 200 eyes", phase="P2")
        MD_EXT = MechData("SetC", T_EXT.sample_id, T_EXT.cluster_id, Y_EXT, T_EXT, S_EXT)
        SANITY_EXT = bootstrap_sanity(MD_EXT, "P2", RNG_SEED + 42)
        write_csv(SANITY_EXT, f"{LOGS}/bootstrap_sanity_external.csv")
        gate("G12_ExternalBootstrapSanity", SANITY_EXT.passed.all() and MD_EXT.n_units == N_EXT_PATIENTS,
             f"{int(SANITY_EXT.passed.sum())}/{len(SANITY_EXT)} checks; resampling unit = patient "
             f"({MD_EXT.n_units}), both eyes kept together", phase="P2")
        STATS_EXT, SEEDCONS_EXT, SPEAR_EXT, P_EXT, DR_EXT = mechanism_stats(
            MD_EXT, RNG_SEED + 50, RNG_SEED + 60, "P2", "DeepDRiD Set-C (exploratory)", "patient")
        STATS_EXT["analysis"] = "patient-clustered hierarchical paired bootstrap, POST-HOC EXPLORATORY"
        write_csv(STATS_EXT, f"{TABLES}/table_mechanism_external_stats.csv")
        write_csv(SEEDCONS_EXT, f"{TABLES}/table_mechanism_seed_consistency_external.csv")
        write_csv(SPEAR_EXT, f"{TABLES}/table_mechanism_external_spearman_sensitivity.csv")

        # ---- exploratory external QWK of all four core conditions (frozen thresholds) ----
        def qwk_from_counts(O, K=NUM_CLASSES):
            w = (np.arange(K)[:, None] - np.arange(K)[None, :]) ** 2 / (K - 1) ** 2
            n = O.sum((-1, -2))
            E = O.sum(-1)[..., :, None] * O.sum(-2)[..., None, :] / np.maximum(n, 1)[..., None, None]
            den = (w * E).sum((-1, -2))
            return np.where(den > 0, 1 - (w * O).sum((-1, -2)) / np.where(den > 0, den, 1), 0.0)
        PRED_EXT = {(c, s): grade_from(S_EXT[(c, s)][head_cols("p_dual")].to_numpy(), THRESHOLDS[c])
                    for c in CORE_CONDITIONS for s in SEEDS_CORE}
        _qrows = [{"model": "Teacher", "condition": "teacher", "threshold": TEACHER_THRESHOLD, "n_seeds": 1,
                   "QWK_mean": _t_ext, "QWK_sd": np.nan, "per_seed": f"{_t_ext:.4f}"}]
        QS = np.array([[fast_qwk(Y_EXT, PRED_EXT[(c, s)]) for s in SEEDS_CORE] for c in CORE_CONDITIONS])
        for i, c in enumerate(CORE_CONDITIONS):
            mets = [all_metrics(Y_EXT, PRED_EXT[(c, s)])[0] for s in SEEDS_CORE]
            _qrows.append({"model": LABEL[c], "condition": c, "threshold": THRESHOLDS[c], "n_seeds": 5,
                           "QWK_mean": QS[i].mean(), "QWK_sd": QS[i].std(ddof=1),
                           "MacroF1_mean": np.mean([m["MacroF1"] for m in mets]),
                           "SER_mean": np.mean([m["SevereErrorRate"] for m in mets]),
                           "per_seed": "; ".join(f"{v:.4f}" for v in QS[i])})
        EXT_QWK_TABLE = pd.DataFrame(_qrows)
        EXT_QWK_TABLE["note"] = ("exploratory; CSD row = published external FP32 CSD result (unchanged); "
                                 "no external winner is selected and deployment is unchanged")
        W_EXT = DR_EXT["U"][:, MD_EXT.unit_of_eye]
        Q_rep = np.empty((B_FINAL, len(CORE_CONDITIONS), len(SEEDS_CORE)))
        for i, c in enumerate(CORE_CONDITIONS):
            for j, s in enumerate(SEEDS_CORE):
                oh = np.zeros((N_EXT_EYES, NUM_CLASSES * NUM_CLASSES))
                oh[np.arange(N_EXT_EYES), Y_EXT * NUM_CLASSES + PRED_EXT[(c, s)]] = 1
                Q_rep[:, i, j] = qwk_from_counts((W_EXT @ oh).reshape(-1, NUM_CLASSES, NUM_CLASSES))
        # spot-check the vectorised QWK against fast_qwk on literally resampled eyes
        _qerr = 0.0
        for b_idx in range(10):
            eyes = np.concatenate([MD_EXT.eyes_of_unit[u] for u in DR_EXT["r"][b_idx]])
            _qerr = max(_qerr, abs(Q_rep[b_idx, 0, 0] - fast_qwk(Y_EXT[eyes], PRED_EXT[(CORE_CONDITIONS[0], SEEDS_CORE[0])][eyes])))
        gate("G12b_QWKBootstrapExact", _qerr < 1e-9, f"vectorised resampled QWK == fast_qwk (max |diff| {_qerr:.1e})",
             phase="P2")
        Qc = combine_seeds(Q_rep, DR_EXT["Sc"])
        _qp = []
        for b in BASELINES:
            e = Qc[:, CORE_CONDITIONS.index(CSD)] - Qc[:, CORE_CONDITIONS.index(b)]
            point = float(np.mean(QS[CORE_CONDITIONS.index(CSD)] - QS[CORE_CONDITIONS.index(b)]))
            s = summarise(e, point)
            _qp.append({"contrast": f"CSD - {LABEL[b]}", "metric": "QWK", **s,
                        "ci_excludes_zero": bool(s["ci_low"] > 0 or s["ci_high"] < 0), "B": B_FINAL,
                        "unit": "patient", "role": "exploratory; predictive, not a mechanism metric"})
        EXT_QWK_PAIRED = pd.DataFrame(_qp)
        write_csv(EXT_QWK_TABLE, f"{TABLES}/table_external_core_qwk_exploratory.csv")
        write_csv(EXT_QWK_PAIRED, f"{TABLES}/table_external_core_qwk_paired_exploratory.csv")

        # ---- rank changes internal vs external, and the teacher's own shift profile ----
        _rk = []
        for m in METRICS:
            mi = {c: float(P_INT[m][MD_INT.ci(c)].mean()) for c in CORE_CONDITIONS}
            me = {c: float(P_EXT[m][MD_EXT.ci(c)].mean()) for c in CORE_CONDITIONS}
            asc = BETTER[m] == "lower"
            ri = {c: i + 1 for i, c in enumerate(sorted(mi, key=mi.get, reverse=not asc))}
            re_ = {c: i + 1 for i, c in enumerate(sorted(me, key=me.get, reverse=not asc))}
            for c in CORE_CONDITIONS:
                _rk.append({"metric": m, "condition": LABEL[c], "internal_mean": mi[c], "internal_rank": ri[c],
                            "external_mean": me[c], "external_rank": re_[c], "rank_change": re_[c] - ri[c]})
        RANKS = pd.DataFrame(_rk)
        write_csv(RANKS, f"{TABLES}/table_mechanism_rank_internal_vs_external.csv")
        def teacher_profile(md, Tdf, y, label):
            Tn = heads_np(Tdf)
            gm = fast_qwk(y, grade_from(Tn["p_macula"], TEACHER_THRESHOLD))
            gd = fast_qwk(y, grade_from(Tn["p_disc"], TEACHER_THRESHOLD))
            gq = fast_qwk(y, grade_from(Tn["p_dual"], TEACHER_THRESHOLD))
            l1 = np.abs(md.dT).sum(1)
            return {"dataset": label, "n_eyes": len(y), "teacher_QWK_dual": gq, "teacher_QWK_aux_macula": gm,
                    "teacher_QWK_aux_disc": gd, "teacher_G_aux": gq - max(gm, gd),
                    "mean_L1_deltaT": float(l1.mean()), "median_L1_deltaT": float(np.median(l1)),
                    "mean_benefitT": float(md.BT.mean()), "frac_benefitT_positive": float((md.BT > 0).mean())}
        TPROF = pd.DataFrame([teacher_profile(MD_INT, T_INT, Y_INT, "DRTiD internal test"),
                              teacher_profile(MD_EXT, T_EXT, Y_EXT, "DeepDRiD Set-C")])
        TPROF["note"] = "G_aux here is a test/external descriptive quantity; the published +0.0469 is on validation"
        write_csv(TPROF, f"{TABLES}/table_teacher_shift_profile.csv")

        CAP_IE = (
            "Gambar suplementer. Selisih fidelitas mekanisme CSD terhadap tiga pembanding pada data uji internal DRTiD "
            "(lingkaran; unit resampling examined-eye record, n = 550) dan DeepDRiD Set-C (persegi; unit resampling pasien, "
            f"100 pasien/200 mata), keduanya dengan CI 95% hierarchical paired bootstrap (B = {B_TXT}) atas lima seed "
            "berpasangan. Titik terisi: CI tidak memuat nol. Analisis Set-C bersifat eksploratori pasca-hoc; model, "
            "ambang dan urutan field (_1 = makula) tidak diubah.\n\n"
            "Supplementary figure. CSD-minus-baseline mechanism-fidelity effects, internal (circles, record-level units) vs "
            "DeepDRiD Set-C (squares, patient-clustered), 95% hierarchical paired bootstrap CIs over five matched seeds. "
            "The external analysis is post-hoc exploratory.")
        forest([STATS_INT, STATS_EXT], ["DRTiD internal (eye records)", "Set-C external (patients)"], ["o", "s"],
               "fig_internal_external_mechanism", CAP_IE)
        PHASE_STATUS["P2"] = "PASSED"
        stage_report("P2", "External mechanism generalisation (exploratory)",
                     [f"Set-C: {N_EXT_PATIENTS} patients / {N_EXT_EYES} eyes / {2 * N_EXT_EYES} images; "
                      f"teacher QWK {_t_ext:.4f}, CSD mean {_csd_mean:.4f} reproduced",
                      f"{int(STATS_EXT.conclusive_favours_csd.sum())}/9 external CIs exclude 0 in CSD's favour, "
                      f"{int((STATS_EXT.ci_excludes_zero & ~STATS_EXT.point_favours_csd).sum())}/9 in a baseline's favour",
                      "patient clusters never split (sanity-checked)"])
        print(STATS_EXT[_show].round(4).to_string(index=False))
        print(EXT_QWK_TABLE.round(4).to_string(index=False))
    except SupplementaryBlocked as e:
        PHASE_STATUS["P2"] = "BLOCKED"; P2_BLOCKER = str(e)
        print("P2 BLOCKED (internal results stand; external analysis is not reported):", e)
print("phase status:", PHASE_STATUS)

## 12 — P3 · Mechanistic decomposition and error analysis (cache only, no new inference)
Exact identity, per record and per threshold:
`pS_dual − pT_dual = (pS_agg − pT_agg) + (ΔS − ΔT)`. Hence `|pS_dual − pT_dual|₁ ≤ |pS_agg − pT_agg|₁ + ShiftL1`:
matching the teacher's **shift** constrains only one of the two terms that determine how close the student's final
cumulative output — and therefore its grade — is to the teacher's. This is algebra, not a causal claim.

In [ ]:
def decomposition(md, Sdict, Tdf, dataset_label, preds_thr):
    Tn = heads_np(Tdf.set_index(Tdf.sample_id.astype(str)).loc[md.ids].reset_index(drop=True))
    aggT = (Tn["p_macula"] + Tn["p_disc"]) / 2
    rows, worst, tri_bad = [], 0.0, 0
    for i, c in enumerate(md.conds):
        for j, s in enumerate(md.seeds):
            d = Sdict[(c, s)]; d = d.set_index(d.sample_id.astype(str)).loc[md.ids].reset_index(drop=True)
            Sn = heads_np(d); aggS = (Sn["p_macula"] + Sn["p_disc"]) / 2
            dual_gap, base_gap = Sn["p_dual"] - Tn["p_dual"], aggS - aggT
            shift_gap = md.dS[i, j] - md.dT
            worst = max(worst, float(np.abs(dual_gap - (base_gap + shift_gap)).max()))
            dl1, bl1, sl1 = np.abs(dual_gap).sum(1), np.abs(base_gap).sum(1), np.abs(shift_gap).sum(1)
            tri_bad += int((dl1 > bl1 + sl1 + 1e-12).sum())
            pred = grade_from(Sn["p_dual"], preds_thr[c])
            m, _ = all_metrics(md.y, pred)
            rows.append({"dataset": dataset_label, "condition": LABEL[c], "seed": s,
                         "base_discrepancy_L1": float(bl1.mean()), "dual_discrepancy_L1": float(dl1.mean()),
                         "ShiftL1": float(sl1.mean()), "ShiftMAE": float(sl1.mean() / NUM_THRESHOLDS),
                         "frac_dual_gap_explained_by_shift": float(sl1.mean() / max(bl1.mean() + sl1.mean(), 1e-12)),
                         "grade_agreement_with_teacher": float((pred == grade_from(Tn["p_dual"], TEACHER_THRESHOLD)).mean()),
                         "QWK": m["QWK"], "MacroF1": m["MacroF1"]})
    return pd.DataFrame(rows), worst, tri_bad

DECOMP, _w, _tb = decomposition(MD_INT, S_INT, T_INT, "DRTiD internal test", THRESHOLDS)
_wx, _tbx = 0.0, 0
if MD_EXT is not None:
    _dx, _wx, _tbx = decomposition(MD_EXT, S_EXT, T_EXT, "DeepDRiD Set-C", THRESHOLDS)
    DECOMP = pd.concat([DECOMP, _dx], ignore_index=True)
gate("G13_DecompositionIdentity", max(_w, _wx) < 1e-12 and _tb + _tbx == 0,
     f"identity max |error| {max(_w, _wx):.1e}; triangle-inequality violations {_tb + _tbx}", phase="P3")
DECOMP_SUM = (DECOMP.groupby(["dataset", "condition"], sort=False)
              [["base_discrepancy_L1", "dual_discrepancy_L1", "ShiftL1", "grade_agreement_with_teacher", "QWK", "MacroF1"]]
              .agg(["mean", "std"]))
DECOMP_SUM.columns = [f"{a}_{b}" for a, b in DECOMP_SUM.columns]
DECOMP_SUM = DECOMP_SUM.reset_index()
write_csv(DECOMP, f"{TABLES}/table_mechanism_decomposition_by_seed.csv")
write_csv(DECOMP_SUM, f"{TABLES}/table_mechanism_decomposition.csv")
# descriptive association across the 20 internal student models (not independent -> no p-value claims)
_di = DECOMP[DECOMP.dataset == "DRTiD internal test"]
ASSOC = pd.DataFrame([{"x": x, "y": "QWK", "spearman_rho_20_models": float(sps.spearmanr(_di[x], _di.QWK)[0]),
                       "note": "descriptive; 20 models share data and teacher -- not independent"}
                      for x in ("ShiftL1", "base_discrepancy_L1", "dual_discrepancy_L1")])
write_csv(ASSOC, f"{TABLES}/table_mechanism_performance_association.csv")
print(DECOMP_SUM.round(4).to_string(index=False))

# ---- per-grade mechanism and errors ----
def by_grade(md, label):
    rows = []
    for g in range(NUM_CLASSES):
        msk = md.y == g
        if not msk.any(): continue
        for c in md.conds:
            i = md.ci(c)
            rows.append({"dataset": label, "true_grade": g, "n_eyes": int(msk.sum()), "condition": LABEL[c],
                         "ShiftL1_mean": float(md.L1[i][:, msk].mean()), "CosAgree_mean": float(md.COS[i][:, msk].mean()),
                         "benefitS_mean": float(md.BS[i][:, msk].mean()), "benefitT_mean": float(md.BT[msk].mean()),
                         "teacher_L1_deltaT_mean": float(np.abs(md.dT[msk]).sum(1).mean())})
    return rows
MECH_GRADE = pd.DataFrame(by_grade(MD_INT, "DRTiD internal test") + (by_grade(MD_EXT, "DeepDRiD Set-C") if MD_EXT is not None else []))
write_csv(MECH_GRADE, f"{TABLES}/table_mechanism_by_grade.csv")

def error_tables(y, preds_by_model, label):
    """preds_by_model: model -> list of prediction arrays (seeds). Rates averaged over seeds."""
    long, summ = [], []
    for model, plist in preds_by_model.items():
        cms = np.stack([confusion_matrix(y, p, labels=list(range(NUM_CLASSES))) for p in plist]).astype(float)
        rates = cms / np.maximum(cms.sum(2, keepdims=True), 1)
        for g in range(NUM_CLASSES):
            for h in range(NUM_CLASSES):
                long.append({"dataset": label, "model": model, "true_grade": g, "pred_grade": h,
                             "support": int(cms[0, g].sum()), "rate_mean": float(rates[:, g, h].mean()),
                             "rate_sd": float(rates[:, g, h].std(ddof=1)) if len(plist) > 1 else np.nan,
                             "count_mean": float(cms[:, g, h].mean())})
        mets = [all_metrics(y, p)[0] for p in plist]
        summ.append({"dataset": label, "model": model, "n_seeds": len(plist),
                     **{k: float(np.mean([m[k] for m in mets])) for k in
                        ("QWK", "MacroF1", "BalancedAccuracy", "SevereErrorRate", "MAE",
                         "Recall_Grade0", "Recall_Grade1", "Recall_Grade2", "Recall_Grade3", "Recall_Grade4")},
                     "G1_to_G0": float(rates[:, 1, 0].mean()), "G1_to_G2": float(rates[:, 1, 2].mean()),
                     "G1_to_G3plus": float(rates[:, 1, 3:].sum(-1).mean()), "support_G1": int(cms[0, 1].sum())})
    return long, summ

_pm_int = {"Teacher": [grade_from(TN_INT["p_dual"], TEACHER_THRESHOLD)]}
_pm_int.update({LABEL[c]: [grade_from(S_INT[(c, s)][head_cols("p_dual")].to_numpy(), THRESHOLDS[c]) for s in SEEDS_CORE]
                for c in CORE_CONDITIONS})
_l, _s = error_tables(Y_INT, _pm_int, "DRTiD internal test")
if MD_EXT is not None:
    _pm_ext = {"Teacher": [grade_from(heads_np(T_EXT)["p_dual"], TEACHER_THRESHOLD)]}
    _pm_ext.update({LABEL[c]: [PRED_EXT[(c, s)] for s in SEEDS_CORE] for c in CORE_CONDITIONS})
    _l2, _s2 = error_tables(Y_EXT, _pm_ext, "DeepDRiD Set-C"); _l += _l2; _s += _s2
ERR_LONG, ERR_SUM = pd.DataFrame(_l), pd.DataFrame(_s)
write_csv(ERR_LONG, f"{TABLES}/table_error_by_grade.csv")
write_csv(ERR_SUM, f"{TABLES}/table_error_summary.csv")
print(ERR_SUM.round(3).to_string(index=False))

# cross-check against the source run's saved confusion matrices (informational)
_cm_bad, _cm_n = [], 0
for c in CORE_CONDITIONS:
    for s in SEEDS_CORE:
        p = f"{O_MET}/confusion_{c}_seed{s}_FP32.csv"
        if os.path.exists(p):
            _cm_n += 1
            ref = pd.read_csv(p, index_col=0).to_numpy()
            mine = confusion_matrix(Y_INT, grade_from(S_INT[(c, s)][head_cols("p_dual")].to_numpy(), THRESHOLDS[c]),
                                    labels=list(range(NUM_CLASSES)))
            if not np.array_equal(ref, mine): _cm_bad.append(f"{c}/s{s}: {int(np.abs(ref - mine).sum())} cells")
gate("G13b_ConfusionMatricesMatchSource", not _cm_bad,
     f"{_cm_n} saved confusion matrices compared" + (f"; differences {_cm_bad[:3]}" if _cm_bad else ", identical"),
     blocking=False, phase="P3")

# ---- teacher-correct vs teacher-incorrect subgroups (exploratory, with uncertainty) ----
_tcorrect = grade_from(TN_INT["p_dual"], TEACHER_THRESHOLD) == Y_INT
_sg = []
for name, msk in (("teacher_correct", _tcorrect), ("teacher_incorrect", ~_tcorrect)):
    sub = MD_INT.subset(msk, f"DRTiD_{name}")
    res, _, P = bootstrap_effects(sub, make_draws(sub.n_units, 5, B_FINAL, RNG_SEED + 70 + int(msk.sum())))
    for (b, m), (e, point) in res.items():
        s = summarise(e, point)
        _sg.append({"subgroup": name, "n_eyes": int(msk.sum()), "contrast": f"CSD - {LABEL[b]}", "metric": m,
                    "csd_mean": float(P[m][sub.ci(CSD)].mean()), "baseline_mean": float(P[m][sub.ci(b)].mean()),
                    **s, "ci_excludes_zero": bool(s["ci_low"] > 0 or s["ci_high"] < 0), "B": B_FINAL,
                    "role": "exploratory subgroup (conditioning on teacher correctness uses labels)"})
SUBGROUP = pd.DataFrame(_sg)
write_csv(SUBGROUP, f"{TABLES}/table_mechanism_teacher_correct_subgroup.csv")
PHASE_STATUS["P3"] = "PASSED"
stage_report("P3", "Decomposition and diagnostics",
             [f"identity pS_dual−pT_dual = (pS_agg−pT_agg) + (ΔS−ΔT) holds to {max(_w, _wx):.1e}",
              "per-grade mechanism/error tables, teacher-correct/incorrect subgroups written"])

## 13 — P4 · Paper-ready outputs (Indonesian), claim ledger, audit and completion marker
Text is generated **from the computed tables** with conditional wording — nothing below is pre-written as a result.

In [ ]:
def md_table(df):
    try:
        return df.to_markdown(index=False)          # needs `tabulate`
    except ImportError:
        return df.to_string(index=False)

def fid(x, d=4, sign=False):
    if x is None or not np.isfinite(x): return "n/a"
    s = f"{x:+.{d}f}" if sign else f"{x:.{d}f}"
    return s.replace("-", "−").replace(".", ",")

def ci_txt(r): return f"[{fid(r.ci_low, sign=True)}; {fid(r.ci_high, sign=True)}]"

def summary_counts(st):
    fav = st[st.conclusive_favours_csd]; unf = st[st.ci_excludes_zero & ~st.point_favours_csd]
    return fav, unf, st[~st.ci_excludes_zero]

def list_id(df):
    items = [f"{r.metric} vs {LABEL[r.baseline]} ({fid(r.effect_point, sign=True)} {ci_txt(r)})" for r in df.itertuples()]
    return "; ".join(items) if items else "-"

fav_i, unf_i, nul_i = summary_counts(STATS_INT)

# ---- Table VI (markdown + csv, compact for 2-column print) ----
def table_vi(st):
    rows = []
    for b in BASELINES:
        r = {"Pembanding": LABEL[b]}
        for m in METRICS:
            x = st[(st.baseline == b) & (st.metric == m)].iloc[0]
            mark = "" if not x.ci_excludes_zero else ("†" if x.point_favours_csd else "‡")
            r[f"Δ{m}"] = f"{fid(x.effect_point, sign=True)}{mark} {ci_txt(x)}"
        rows.append(r)
    return pd.DataFrame(rows)
TAB_VI = table_vi(STATS_INT)
write_csv(TAB_VI, f"{TABLES}/table_VI_paper_internal.csv")
TAB_VI_MD = md_table(TAB_VI)

perm_agree = int(((STATS_INT.p_holm_9 < 0.05) == STATS_INT.ci_excludes_zero).sum())
rec_agree = int((STATS_INT.recordonly_excludes_zero == STATS_INT.ci_excludes_zero).sum())

III_C = (
    "Sebagai analisis suplementer yang bersifat eksploratori karena dirancang setelah hasil utama diketahui, selisih "
    "ketiga metrik shift fidelity antara CSD dan setiap pembanding (ShiftL1, CosAgree, dan BenefitCorr; sembilan kontras) "
    f"diestimasi melalui hierarchical paired bootstrap (B = {B_TXT}). Setiap replikasi meresampling examined-eye records "
    "dan lima core seeds secara bersamaan untuk CSD, pembanding, dan teacher, lalu menghitung ulang setiap metrik, "
    "termasuk korelasi Pearson BenefitCorr, dari record hasil resampling. Uji permutasi berpasangan bersyarat pada tingkat "
    "record dengan koreksi Holm atas sembilan kontras dilaporkan sebagai bukti pelengkap. Seluruh analisis menggunakan "
    "checkpoint, prapemrosesan, dan ambang keputusan yang telah dibekukan tanpa pelatihan maupun penyetelan ulang.")

def para_ivc():
    s = (f"Analisis suplementer eksploratori pada Tabel VI mengestimasi selisih fidelitas mekanisme CSD terhadap ketiga "
         f"pembanding pada 550 mata data uji internal dengan lima seed berpasangan. ")
    if len(fav_i) == 9:
        s += ("Kesembilan CI 95% tidak memuat nol dan seluruhnya berada pada arah yang menguntungkan CSD, sehingga "
              "keunggulan deskriptif CSD pada ShiftL1, CosAgree, dan BenefitCorr didukung secara inferensial dalam "
              "batas analisis ini. ")
    elif len(fav_i) > 0:
        s += (f"Sebanyak {len(fav_i)} dari sembilan CI 95% tidak memuat nol pada arah yang menguntungkan CSD, yaitu "
              f"{list_id(fav_i)}. {len(nul_i)} kontras lainnya memiliki CI yang memuat nol sehingga belum konklusif. ")
    else:
        s += ("Tidak satu pun CI 95% berada sepenuhnya pada arah yang menguntungkan CSD; keunggulan deskriptif CSD "
              "belum didukung secara inferensial. ")
    if len(unf_i):
        s += f"Kontras yang menguntungkan pembanding: {list_id(unf_i)}. "
    s += (f"Arah selisih per seed menguntungkan CSD pada {int(STATS_INT.seeds_favouring_csd.sum())} dari 45 pasangan "
          f"seed-kontras. Bootstrap tingkat record dengan model tetap sejalan dengan CI hierarkis pada {rec_agree} dari "
          f"sembilan kontras, dan uji permutasi terkoreksi Holm sejalan pada {perm_agree} dari sembilan kontras. "
          + (f"Ketidaksesuaian pada {9 - rec_agree} kontras antara bootstrap tingkat record dan CI hierarkis menunjukkan "
             "bahwa variasi antar-seed pelatihan, bukan variasi antar-record, mendominasi ketidakpastian pada kontras "
             "tersebut. " if rec_agree < 9 else "") +
          "Karena hanya lima model terlatih per kondisi, resampling record tidak menambah jumlah model, sehingga interval "
          "ini perlu dibaca sebagai bukti eksploratori. Hasil ini tidak mengubah kesimpulan bahwa keunggulan QWK CSD "
          "belum konklusif (Tabel II).")
    return s

def para_ive():
    if STATS_EXT is None:
        return ("Analisis eksternal fidelitas mekanisme tidak dilaporkan karena tahap P2 tidak lolos gate reproduksi "
                f"atau tidak dijalankan (status: {PHASE_STATUS['P2']}).")
    fav, unf, nul = summary_counts(STATS_EXT)
    s = ("Analisis eksternal tambahan yang bersifat eksploratori pasca-hoc menguji apakah keselarasan prediction shift "
         "yang dipelajari pada DRTiD tetap bertahan pada DeepDRiD Set-C (100 pasien, 200 mata, 400 citra; urutan field "
         "_1 = makula), menggunakan model FP32 dan ambang keputusan yang sama tanpa penyetelan ulang serta bootstrap "
         "berpasangan berklaster pasien. ")
    if len(fav):
        s += f"Sebanyak {len(fav)} dari sembilan CI 95% tidak memuat nol pada arah yang menguntungkan CSD ({list_id(fav)}). "
    else:
        s += "Tidak ada CI 95% yang sepenuhnya berada pada arah yang menguntungkan CSD. "
    if len(unf): s += f"Kontras yang menguntungkan pembanding: {list_id(unf)}. "
    s += f"{len(nul)} kontras memiliki CI yang memuat nol. "
    rk = RANKS[RANKS.condition == "CSD"]
    kept = [r.metric for r in rk.itertuples() if r.external_rank == 1]
    kept_i = [r.metric for r in rk.itertuples() if r.internal_rank == 1]
    s += (f"CSD menempati peringkat deskriptif pertama pada {len(kept)} dari tiga metrik di Set-C"
          + (f" ({', '.join(kept)})" if kept else "") + f", dibandingkan {len(kept_i)} dari tiga pada data internal. ")
    tp = TPROF.set_index("dataset")
    s += (f"Rerata |Δ_T| teacher berubah dari {fid(tp.loc['DRTiD internal test','mean_L1_deltaT'])} (internal) menjadi "
          f"{fid(tp.loc['DeepDRiD Set-C','mean_L1_deltaT'])} (Set-C), sehingga perubahan fidelitas eksternal juga "
          "mencerminkan perubahan target teacher akibat pergeseran domain, bukan hanya kualitas transfer student. "
          "Hasil ini dipisahkan dari QWK eksternal pada Tabel V dan tidak dimaksudkan sebagai validasi klinis.")
    return s

def para_ivf():
    d = DECOMP_SUM[DECOMP_SUM.dataset == "DRTiD internal test"].set_index("condition")
    parts = [f"{c}: ShiftL1 {fid(d.loc[c,'ShiftL1_mean'])}, selisih agregasi dasar {fid(d.loc[c,'base_discrepancy_L1_mean'])}, "
             f"selisih prediksi joint {fid(d.loc[c,'dual_discrepancy_L1_mean'])}" for c in ("CSD", "Feature-KD")]
    g1 = ERR_SUM[ERR_SUM.dataset == "DRTiD internal test"].set_index("model")
    return ("Secara aljabar, p_dual^S − p_dual^T = (p_agg^S − p_agg^T) + (Δ_S − Δ_T) untuk setiap record "
            "(terverifikasi numerik; galat maksimum < 1e−12). Keselarasan Δ yang lebih tinggi hanya mengendalikan "
            "suku kedua; suku agregasi dasar tetap dapat berbeda, sehingga fidelitas mekanisme yang lebih baik tidak secara "
            f"matematis menjamin grade akhir yang lebih tepat. Pada data uji internal (rerata lima seed, norma L1) — {'; '.join(parts)}. "
            f"Kelemahan Grade 1 bersifat umum: recall Grade 1 teacher {fid(g1.loc['Teacher','Recall_Grade1'],3)}, CSD "
            f"{fid(g1.loc['CSD','Recall_Grade1'],3)}, Feature-KD {fid(g1.loc['Feature-KD','Recall_Grade1'],3)}; pada CSD, "
            f"{fid(100*g1.loc['CSD','G1_to_G0'],1)}% mata Grade 1 diprediksi Grade 0 dan {fid(100*g1.loc['CSD','G1_to_G2'],1)}% "
            "Grade 2. Analisis ini deskriptif dan tidak menetapkan hubungan kausal.")

def para_v():
    s = "RQ1 (tambahan). "
    if len(fav_i) == 9:
        s += "Analisis suplementer eksploratori mendukung secara inferensial keunggulan fidelitas mekanisme CSD terhadap ketiga pembanding pada ketiga metrik. "
    elif len(fav_i) > 0:
        s += (f"Analisis suplementer eksploratori mendukung keunggulan fidelitas mekanisme CSD pada {len(fav_i)} dari sembilan "
              "kontras, sedangkan kontras lainnya belum konklusif. ")
    else:
        s += "Analisis suplementer eksploratori belum memberikan dukungan inferensial bagi keunggulan deskriptif fidelitas mekanisme CSD. "
    if STATS_EXT is not None:
        fe = int(STATS_EXT.conclusive_favours_csd.sum())
        s += f"Pada DeepDRiD Set-C, {fe} dari sembilan kontras eksploratori mempertahankan dukungan tersebut. "
    s += "Kesimpulan prediktif RQ1, pemilihan QAT sebagai artefak deployment, dan keunggulan PTQ pada QWK eksternal tidak berubah."
    return s

CAP_TAB_VI = ("TABEL VI — SELISIH FIDELITAS MEKANISME CSD TERHADAP TIGA PEMBANDING PADA DATA UJI INTERNAL DRTID "
              "(ANALISIS SUPLEMENTER EKSPLORATORI; RERATA LIMA SEED BERPASANGAN, CI 95% HIERARCHICAL PAIRED BOOTSTRAP, "
              f"B = {B_TXT}). *Catatan: Δ = CSD − pembanding. ShiftL1 lebih rendah lebih baik; CosAgree dan BenefitCorr lebih "
              "tinggi lebih baik. † CI tidak memuat nol dan menguntungkan CSD; ‡ CI tidak memuat nol dan menguntungkan "
              "pembanding. Unit resampling: examined-eye record (n = 550) dan seed.")

ext_block = ""
if STATS_EXT is not None:
    TAB_VI_EXT = table_vi(STATS_EXT)
    write_csv(TAB_VI_EXT, f"{TABLES}/table_VI_paper_external_optional.csv")
    ext_block = ("\n### Tabel opsional (Set-C, eksploratori, berklaster pasien)\n\n"
                 + md_table(TAB_VI_EXT)
                 + "\n\n**QWK eksternal eksploratori (ambang beku):**\n\n"
                 + md_table(pd.DataFrame({"Model": EXT_QWK_TABLE.model,
                                          "Ambang": [fid(x, 2) for x in EXT_QWK_TABLE.threshold],
                                          "Seed": EXT_QWK_TABLE.n_seeds,
                                          "QWK (rerata ± SB)": [fid(a) + ("" if not np.isfinite(b) else " ± " + fid(b))
                                                                for a, b in zip(EXT_QWK_TABLE.QWK_mean, EXT_QWK_TABLE.QWK_sd)]}))
                 + "\n")

PAPER_MD = f"""# DR-VERGE — Hasil suplementer siap-tempel (Bahasa Indonesia)

> Dihasilkan otomatis dari tabel di `{TABLES}`. Semua angka berasal dari analisis yang benar-benar dijalankan pada
> sesi ini; analisis bersifat **eksploratori pasca-hoc**. Jangan mengubah angka secara manual.

## III.C — tambahan 2–3 kalimat (setelah paragraf bootstrap QWK)

{III_C}

## IV.C — Tabel VI + paragraf

{CAP_TAB_VI}

{TAB_VI_MD}

{para_ivc()}

## IV.E — paragraf eksternal (eksploratori)

{para_ive()}
{ext_block}
## IV.F (opsional) — Analisis diagnostik dan implikasi

{para_ivf()}

## V — kalimat tambahan kesimpulan

{para_v()}

## Keterangan gambar

**fig_mechanism_effect_forest** — {CAP_FOREST_INT.split(chr(10))[0]}

**fig_internal_external_mechanism** — {"(tidak dihasilkan; P2 " + PHASE_STATUS['P2'] + ")" if STATS_EXT is None else CAP_IE.split(chr(10))[0]}
"""
write_text(f"{REPORTS}/PAPER_READY_RESULTS_ID.md", PAPER_MD)

# ---- claim ledger ----
CL = []
def claim(text, supported, evidence, source):
    CL.append({"claim": text, "status": supported, "evidence": evidence, "source_file": source})
claim("Original Enhanced teacher/student QWK and mechanism means are reproduced from frozen checkpoints",
      "SUPPORTED", f"max |diff| {REPRO_DF.abs_diff.max():.1e}", "logs/reproduction_checks.csv")
claim("CSD is descriptively best on ShiftL1, CosAgree and BenefitCorr (internal)",
      "SUPPORTED" if all((MEANS_INT.loc[CSD, m] == (MEANS_INT[m].min() if BETTER[m] == "lower" else MEANS_INT[m].max()))
                         for m in METRICS) else "NOT SUPPORTED",
      "; ".join(f"{m} CSD {MEANS_INT.loc[CSD, m]:.4f}" for m in METRICS), "tables/table_reproduced_seed_metrics_internal.csv")
for r in STATS_INT.itertuples():
    claim(f"Internal: {r.metric} favours CSD over {LABEL[r.baseline]} (95% CI excludes 0)",
          "SUPPORTED (exploratory)" if r.conclusive_favours_csd else ("CONTRADICTED" if r.ci_excludes_zero else "NOT CONCLUSIVE"),
          f"{r.effect_point:+.4f} [{r.ci_low:+.4f}, {r.ci_high:+.4f}], seeds {r.seeds_favouring_csd}/5, p_holm {r.p_holm_9:.4f}",
          "tables/table_mechanism_internal_stats.csv")
if STATS_EXT is not None:
    for r in STATS_EXT.itertuples():
        claim(f"Set-C: {r.metric} favours CSD over {LABEL[r.baseline]} (patient-clustered CI excludes 0)",
              "SUPPORTED (post-hoc exploratory)" if r.conclusive_favours_csd else
              ("CONTRADICTED" if r.ci_excludes_zero else "NOT CONCLUSIVE"),
              f"{r.effect_point:+.4f} [{r.ci_low:+.4f}, {r.ci_high:+.4f}]", "tables/table_mechanism_external_stats.csv")
claim("CSD improves QWK over the baselines", "NOT SUPPORTED (unchanged)", "published Table II CIs all include 0",
      "original run")
claim("Higher Δ agreement mathematically guarantees a better final grade", "NOT SUPPORTED",
      "identity: dual gap = base gap + shift gap", "tables/table_mechanism_decomposition.csv")
claim("Results establish clinical readiness / clinical validation", "NOT SUPPORTED", "computational evaluation only", "-")
claim("Δ measures anatomical complementarity", "NOT SUPPORTED",
      "operational joint-vs-individual prediction-shift proxy", "source notebook §8")
claim("Internal resampling is patient-clustered", "NOT SUPPORTED", "DRTiD exposes no patient key: unit = examined-eye record",
      "source notebook §4")
CLAIMS = pd.DataFrame(CL)
write_csv(CLAIMS, f"{REPORTS}/CLAIM_LEDGER.csv")

LIMITS = f"""# Exploratory status and limitations

- All supplementary inference was **designed after** the main Enhanced results were known: it is exploratory/post-hoc,
  even where intervals exclude zero. Nothing here is preregistered or confirmatory.
- Only **five trained models per condition** exist. Resampling records/patients does not create new trained models;
  seed-level uncertainty rests on five values (seed effect directions are reported in
  `table_mechanism_seed_consistency.csv`, leave-one-seed-out ranges in `table_mechanism_internal_stats.csv`).
- The 9 internal CIs are **not multiplicity-adjusted**. The conditional permutation p-values are Holm-adjusted over
  the 9 contrasts as one family, but that test conditions on the trained models and ignores seed variance.
- Δ is an **operational joint-vs-individual cumulative prediction-shift proxy** produced by three heads; it carries head
  and calibration differences and is not a measurement of anatomical complementarity.
- BenefitCorr is a correlation of per-record NLL benefits, **not** an accuracy gain; ordinal sigmoid outputs trained with
  weighted BCE are threshold scores, not calibrated clinical probabilities.
- DRTiD exposes no patient key: internal units are **examined-eye records**, never described as patients. Only Set-C
  (100 patients) supports patient clustering; 100 patients limit external precision.
- External label semantics and domain shift change the teacher's own shift target (see `table_teacher_shift_profile.csv`),
  so external fidelity mixes target change with student transfer.
- The external four-condition QWK table is exploratory; the published CSD external QWK is unchanged, no external winner is
  selected and the QAT deployment decision is unchanged.
- Teacher-correct/incorrect subgroups condition on labels and are descriptive/exploratory.
- Different GPU numerics (e.g. TF32 on A100 vs FP32 elsewhere) can shift probabilities by ~1e-4; reproduction tolerances
  and observed differences are in `logs/reproduction_checks.csv`.
- P2 status in this run: **{PHASE_STATUS['P2']}**{(' — ' + P2_BLOCKER) if P2_BLOCKER else ''}.
"""
write_text(f"{REPORTS}/EXPLORATORY_LIMITATIONS.md", LIMITS)

# ---- final integrity: nothing in the source tree changed, no optimizer ever existed ----
_after = {k: sha256_file(r.path) for k, r in zip(CKPT_MANIFEST.key, CKPT_MANIFEST.itertuples())}
_ck_changed = [k for k in CKPT_SHA if _after.get(k) != CKPT_SHA[k]]
_ref_changed = [k for k, v in REF_SHA.items() if sha256_file(INV_DF.set_index("key").loc[k, "path"]) != v]
gate("G14_SourceUnchanged", not _ck_changed and not _ref_changed,
     f"{len(CKPT_SHA)} checkpoints and {len(REF_SHA)} source files re-hashed: identical" if not (_ck_changed or _ref_changed)
     else f"CHANGED: {_ck_changed + _ref_changed}", phase="P4")
_opts = [o for o in gc.get_objects() if isinstance(o, torch.optim.Optimizer)]
gate("G14b_NoOptimizerEverCreated", not _opts and not torch.is_grad_enabled(),
     "no torch.optim.Optimizer object exists; autograd disabled throughout", phase="P4")

AUDIT = f"""# Reproduction audit

**Source run:** `{ORIG_ART}` (read-only) · **Supplementary outputs:** `{SUPP_ART}`
**Device:** {DEVICE} ({ENVIRONMENT['gpu']}) · torch {torch.__version__} · albumentations {A.__version__}
**Source run environment:** {ENVIRONMENT['original_run_environment']}
**Source notebook:** {SOURCE_NB}
**Random seeds:** draws derive from RNG_SEED={RNG_SEED}; internal draw hash `{DR_INT['hash'][:16]}`
{('; external draw hash `' + DR_EXT['hash'][:16] + '`') if STATS_EXT is not None else ''}

## Phase status
{json.dumps(PHASE_STATUS, indent=2)}

## Reproduced internal values (5-seed means; teacher single seed)
Teacher QWK {_tq:.6f} (source {TEACHER_INT['ref_QWK']})

{MEANS_INT[['QWK', 'ShiftL1', 'CosAgree', 'BenefitCorr', 'BenefitCorrSpearman']].round(6).to_string()}

Tolerances: {json.dumps(TOL)}
Largest absolute difference across {len(REPRO_DF)} reproduction checks: {REPRO_DF.abs_diff.max():.2e}

## Gates
{pd.DataFrame(GATES)[['phase', 'gate', 'passed', 'blocking', 'detail']].to_string(index=False)}
"""
write_text(f"{REPORTS}/REPRODUCTION_AUDIT.md", AUDIT)
write_json({"config": {"B_PILOT": B_PILOT, "B_FINAL": B_FINAL, "P_PERM": P_PERM, "RNG_SEED": RNG_SEED,
                       "INFER_DEVICE": INFER_DEVICE, "RUN_EXTERNAL": RUN_EXTERNAL, "TOL": TOL,
                       "thresholds": THRESHOLDS, "teacher_threshold": TEACHER_THRESHOLD},
            "phase_status": PHASE_STATUS, "source_notebook": SOURCE_NB, "checkpoint_sha256": CKPT_SHA,
            "reference_file_sha256": REF_SHA, "environment": ENVIRONMENT,
            "draw_hash_internal": DR_INT["hash"], "draw_hash_external": DR_EXT["hash"] if STATS_EXT is not None else None,
            "inference": INFER_LOG}, f"{LOGS}/run_manifest.json")

PHASE_STATUS["P4"] = "PASSED"
_blocking_failed = [g for g in GATES if g["blocking"] and not g["passed"] and g["phase"] != "P2"]
_done_ok = (not _blocking_failed and all(PHASE_STATUS[p] == "PASSED" for p in ("P0", "P1", "P3", "P4"))
            and PHASE_STATUS["P2"] in ("PASSED", "SKIPPED_BY_CONFIG"))
_marker = "SUPPLEMENTARY_RUN_COMPLETE.txt" if _done_ok else "SUPPLEMENTARY_RUN_PARTIAL.txt"
write_text(f"{LOGS}/{_marker}",
           f"completed_at={time.strftime('%Y-%m-%dT%H:%M:%S')}\nphase_status={json.dumps(PHASE_STATUS)}\n"
           f"gates_passed={sum(g['passed'] for g in GATES)}/{len(GATES)}\nsource_unchanged=True\n"
           f"device={DEVICE}\nB_FINAL={B_FINAL}\n")
stage_report("P4", "Paper-ready outputs", [f"{_marker} written", "reports/PAPER_READY_RESULTS_ID.md",
                                          "reports/CLAIM_LEDGER.csv", "reports/REPRODUCTION_AUDIT.md"])
print(f"\n{_marker} written | gates passed {sum(g['passed'] for g in GATES)}/{len(GATES)} | {PHASE_STATUS}")

## 14 — Summary

In [ ]:
print("=" * 100)
print("INTERNAL (DRTiD test, 550 eye records, 5 matched seeds) — CSD minus baseline, 95% hierarchical CI")
print(STATS_INT[_show].round(4).to_string(index=False))
if STATS_EXT is not None:
    print("\nEXTERNAL (DeepDRiD Set-C, 100 patients) — exploratory, patient-clustered")
    print(STATS_EXT[_show].round(4).to_string(index=False))
    print("\nExploratory external QWK (frozen thresholds):")
    print(EXT_QWK_TABLE[["model", "threshold", "QWK_mean", "QWK_sd", "per_seed"]].to_string(index=False))
print("\nDecomposition (means over seeds):")
print(DECOMP_SUM[["dataset", "condition", "base_discrepancy_L1_mean", "ShiftL1_mean", "dual_discrepancy_L1_mean",
                  "QWK_mean"]].round(4).to_string(index=False))
print("\nOutputs:", SUPP_ART)
try:
    from IPython.display import Image as _Img, display as _display
    for _f in ("fig_mechanism_effect_forest.png", "fig_internal_external_mechanism.png"):
        if os.path.exists(f"{FIGS}/{_f}"): _display(_Img(filename=f"{FIGS}/{_f}", width=900))
except Exception as _e:
    print("(figures saved; inline preview unavailable:", _e, ")")
for d in (TABLES, FIGS, REPORTS, LOGS):
    print(" ", os.path.relpath(d, SUPP_ART) + "/", sorted(os.listdir(d))[:40])